# 🎓 RAG 기반 졸업요건 질의응답 챗봇
### 순천향대학교 SW융합대학 「2026 학과별 다면적 졸업인증제」

**사용법:** 상단 메뉴 **런타임 → 모두 실행**(Ctrl+F9). 마지막 셀에 챗봇 화면과 공개 링크(`https://….gradio.live`)가 나옵니다.

| 과제 요구사항 | 구현 |
|---|---|
| 1. 다면적졸업인증제 PDF 넣기 | PDF 원문 청크 + PDF의 표를 구조화한 `requirements.json` 청크 |
| 2. 안내받은 다른 문서도 넣기 | `docs/` 폴더의 PDF·TXT·MD·DOCX 자동 색인 (아래 '문서 추가' 셀) |
| 3. 활동 체크 + 수업 리스트 → 졸업 충족 판정 | ① 자가진단 탭(체크박스·횟수·과목 리스트) ② 채팅에서 말로 설명 → LLM이 `check_graduation` 도구 호출 |

```
문서(PDF/TXT/DOCX) ─► 청크 분할 ─┐
requirements.json ─► 구조화 청크 ─┴─► 하이브리드 검색(TF-IDF + 임베딩 + 학과/학번 부스팅)
                                                  │
질문 + 학과/학번 + 자가진단 결과 ───────────────► LLM ──tool calling──► 졸업요건 규칙 엔진
```

## 0. 설치 및 API 키
아래 셀에서 **PROVIDER** 를 고르고 실행하면 API 키 입력창이 뜹니다. (입력한 키는 노트북에 저장되지 않음)
- **Gemini (무료)**: https://aistudio.google.com → API keys → Create API key
- **OpenAI (유료, 최소 $5 충전)**: https://platform.openai.com → API keys → Create new secret key

**키 없이 엔터만 치면 '오프라인 모드'** (LLM 없이 검색 결과 + 자가진단만 동작).

In [ ]:
!pip -q install "openai>=1.40" "gradio>=5.0,<6.0" pypdf python-docx
!mkdir -p rag data docs index

In [ ]:
import os
from getpass import getpass
PROVIDER = "Gemini (무료)"  #@param ["Gemini (무료)", "OpenAI (유료)"]
secret_name = "GEMINI_API_KEY" if PROVIDER.startswith("Gemini") else "OPENAI_API_KEY"
key = ""
try:
    from google.colab import userdata          # 왼쪽 🔑 보안 비밀에 저장해 두었으면 자동 사용
    key = userdata.get(secret_name) or ""
except Exception:
    pass
if not key:
    key = getpass(f"{secret_name} (없으면 엔터 → 오프라인 모드): ").strip()
if key:
    os.environ["OPENAI_API_KEY"] = key         # 코드는 OpenAI 호환 방식으로 두 회사 모두 지원
if PROVIDER.startswith("Gemini"):
    os.environ["OPENAI_BASE_URL"] = "https://generativelanguage.googleapis.com/v1beta/openai/"
    os.environ["CHAT_MODEL"] = "gemini-3.8-flash"
    os.environ["EMBED_MODEL"] = "gemini-embedding-2"
else:
    os.environ.pop("OPENAI_BASE_URL", None)
    os.environ["CHAT_MODEL"] = "gpt-4o-mini"
    os.environ["EMBED_MODEL"] = "text-embedding-3-small"
print("제공자:", PROVIDER, "| 모델:", os.environ["CHAT_MODEL"], "| LLM 모드:", "ON" if key else "OFF (오프라인)")

## 1. 문서 넣기
### 1-1. 다면적졸업인증제 PDF (노트북에 내장)

In [ ]:
#@title 📄 다면적졸업인증제 PDF를 docs/ 에 저장 (코드 숨김)
import base64
PDF_B64 = "..."
with open("docs/2026_다면적졸업인증제_SW융합대학.pdf", "wb") as f:
    f.write(base64.b64decode(PDF_B64))
print("저장 완료: docs/2026_다면적졸업인증제_SW융합대학.pdf")

### 1-2. 순천향대 공식 안내 문서 (학교 홈페이지에서 수집)
다면적졸업인증제와 관련된 순천향대 공식 공지·안내 페이지 내용을 정리한 문서입니다.
졸업요건 구성(학칙 제33조), SW 기초교육 의무, **AI·SW 전공역량 졸업인증제(200점) 점수표**, **코딩역량 인증제**, TOPCIT 혜택, 비교과 점수 프로그램 예시, ASP·多Dream비교과센터 안내가 들어 있고 항목마다 출처 URL이 있습니다.

In [ ]:
%%writefile docs/순천향대_졸업인증_관련_공식안내.md
# 순천향대학교 졸업인증 관련 공식 안내 모음

순천향대학교 공식 홈페이지 공지와 SW중심대학사업단 안내 페이지에서 다면적 졸업인증제와 관련된 내용을 모아 정리한 문서입니다. (자료 확인일: 2026-11-08)
각 항목 끝에 출처를 적었습니다. 세부 기준은 바뀔 수 있으므로 최종 확인은 출처 페이지와 차세대 종합정보시스템에서 하세요.

## 1. 졸업요건의 구성 (다면적졸업인증은 졸업요건 중 하나)

- 순천향대학교 졸업요건에는 졸업학점, 졸업종합평가(시험/논문/작품 등), 다면적졸업인증, 봉사활동 등이 포함됩니다. 즉 다면적졸업인증 점수를 모두 채워도 졸업학점·졸업종합평가 등 다른 요건을 함께 충족해야 졸업할 수 있습니다.
- 졸업요건의 근거는 학칙 제33조(졸업요건)입니다. 학칙에는 130학점 이상 취득(일부 학과·과정은 별도 학점 기준), 졸업논문(졸업종합시험, 졸업작품 등) 심사 합격, 교육단위별 졸업요건 충족 등이 규정되어 있고, 그 밖의 교육단위별 졸업요건은 따로 정합니다.
- 졸업에 필요한 학점은 취득했지만 다른 졸업요건(예: 다면적졸업인증)을 충족하지 못하면 학사과정 수료로 처리되고, 이후 모든 졸업요건을 충족한 학기의 졸업일에 학위가 수여됩니다.
- 본인의 졸업사정 결과는 포털사이트 → 학사인트라넷 → 학부학사행정에서 확인할 수 있습니다.

출처: 순천향대학교 대학공지 「2026년 8월 졸업대상자 학사학위 취득유예(졸업유예) 신청 안내」(2026.07) https://home.sch.ac.kr/eng/03/0201.jsp?mode=view&article_no=20260715112125254005 / 순천향대학교 학칙 제33조(규정관리시스템 rule.sch.ac.kr)

## 2. 전교생 SW 기초교육 이수 의무

- 순천향대학교는 전교생에게 SW 기초교육을 의무로 운영합니다.
- 공통교양 '컴퓨터코딩과문제해결'(2학점)과 중핵교양 'SW와AI' 영역(3학점)을 합쳐 총 5학점을 이수해야 합니다.
- SW중심대학사업단은 SW학과 전공 이수학점을 93학점으로 확대 개편했다고 안내하고 있습니다. (학과·학번별 정확한 전공 이수학점은 학과 교육과정표에서 확인)

출처: 순천향대학교 직속기관 SW중심대학사업단 소개 https://home.sch.ac.kr/sch/04/000600.jsp

## 3. AI·SW 전공역량 졸업인증제 (전공역량점수 200점)

SW융합대학 학과의 다면적 졸업인증 기준표에 있는 'AI·SW전공역량졸업인증제' 항목의 세부 기준입니다. (의료IT공학과는 표에 '200점 이상'으로 표기)

- 목적: SW전공 학생들이 다양한 전공 관련 비교과 활동에 참여하도록 독려하고, 산업체 수요를 반영한 기준을 졸업인증에 반영하기 위한 제도입니다.
- 인증 기준: 아래 활동으로 얻은 전공역량점수 합계가 **200점 이상**이면 인증됩니다.
- 인증요건은 인증자 비율, 취업 후 산업체 만족도, 산업체 수요 분석을 바탕으로 교과과정혁신위원회가 개선합니다.

| 구분 | 인정 조건 | 전공역량점수 |
|---|---|---|
| 전공교과목 학점 | 전공 교과목 5과목을 B학점 이상으로 이수 | 100점 |
| 산학협력프로젝트(캡스톤디자인) | 산업체 평가 '최우수' 또는 '우수' 획득 | 50점 |
| 인턴십 | 국내·외 인턴십 기관 평가 80점 이상 | 국외 100점 / 국내 50점 |
| 공모전·경진대회 | 교내·외 공모전 또는 경진대회 입상 | 교외 50점 / 교내 20점 |
| 오픈소스SW활동 | 교내 오픈소스 SW동아리 활동 | 30점 |
| 창업 | 전공 관련 창업동아리 활동 | 100점 |
| 졸업논문 | 졸업논문 통과 | 100점 |
| 졸업시험 | 졸업시험 80점 이상 | 100점 |
| 연구논문 게재 | 국외·국내 학술지 논문 게재 | 국외 50점 / 국내 30점 |
| 특허 출원 | 국내 특허 출원 | 50점 |
| TOPCIT | TOPCIT 수준2 이상 | 50점 |

- 예시: 전공 교과목 5과목 B 이상(100점) + 졸업논문 통과(100점) = 200점으로 인증 요건을 충족합니다.
- 예시: 전공 교과목 5과목 B 이상(100점) + 교외 경진대회 입상(50점) + TOPCIT 수준2 이상(50점) = 200점.

출처: 순천향대학교 SW중심대학사업단 「AI·SW 전공역량 졸업인증제」 https://home.sch.ac.kr/sw/02/050200.jsp

## 4. 코딩역량 인증제

SW융합대학 학과의 다면적 졸업인증 기준표에 있는 '코딩역량인증제(Level2 이상)' 항목에 해당하는 대학 자체 인증 시험입니다.

- 목적: SW전공 학생들의 실무 코딩 역량을 진단하고 높이기 위해 대학 차원에서 운영합니다.
- 사용 언어: C/C++, JAVA, Python 중 선택
- 시험 분야: 알고리즘
- 문항: 10문항, 100점 만점
- 등급: 레벨1, 레벨2, 레벨3
- 사업단 안내 페이지의 인증요건: 70점 이상
- 다면적 졸업인증 기준표(학과별)에서는 대부분 '코딩역량인증제 Level2 이상'을 요구하므로, 졸업인증 점수(5점)를 받으려면 레벨2 이상을 받아야 합니다. 레벨별 점수 구간은 사업단 공지를 확인하세요.

출처: 순천향대학교 SW중심대학사업단 「코딩역량 인증제」 https://home.sch.ac.kr/sw/02/050300.jsp

## 5. TOPCIT 정기평가 응시 혜택

- TOPCIT 정기평가에 응시한 학생에게는 SW역량 마일리지 5점이 부여됩니다. (2026년 제25회 정기평가 공지 기준)
- 같은 공지에서 다면적졸업인증 점수(창의 1점)는 사업단에서 일괄 부여한다고 안내했습니다. 이는 여러 학과 기준표의 'TOPCIT 응시(회당 1점)'와 같은 배점입니다.
- 성적 우수자와 성적 향상자에게는 장학금이 지급됩니다. 장학금을 받으려면 TOPCIT 홈페이지(www.topcit.or.kr) → MY PAGE → 성적조회/인증서 발급에서 성적인증서를 내려받아 기한 내에 제출해야 합니다.
- TOPCIT 장학금 문의: 041-530-4738

출처: 순천향대학교 SW중심대학사업단 공지 「[AI전공교육센터] 제25회 TOPCIT 성적우수 및 향상 장학금 지급 안내」(2026.07.03) https://home.sch.ac.kr/sw/07/010000.jsp?mode=view&article_no=20260703165638209072

## 6. 비교과자율활동 점수를 받을 수 있는 프로그램 예시 (2026학년도 2학기 공지)

학과 기준표의 '비교과자율활동' 점수는 학교가 운영하는 다양한 비교과 프로그램에 참여해 채울 수 있습니다. 2026학년도 2학기 공지에 나온 예시는 다음과 같습니다. 프로그램과 점수는 학기마다 달라지므로 대학 공지사항을 확인하세요.

- 프로젝트형 다문화 멘토링(향설교양대학 사회공헌나눔센터): 한 학기 10회기 1:1 멘토링과 필수 행사 3회에 참여하면 다면적 졸업인증 공감형 점수 5점과 봉사시간이 인정됩니다. 휴학생은 점수를 받을 수 없습니다.
  출처: https://home.sch.ac.kr/sc/03/0101.jsp?mode=view&article_no=20260819092739830057
- 향설열린강좌 청강(향설교양대학): 사전 청강 신청과 청강 후 서명을 완료한 재학생에게 다면적 졸업인증제 창의 점수 2점이 부여됩니다.
  출처: https://home.sch.ac.kr/carbon/03/0201.jsp?mode=view&article_no=20260810165943620047
- 비교과 프로그램 신청: 차세대 종합정보시스템 → 학부학사행정 → 비교과과정 → 신청관리 → 비교과신청

## 7. ASP와 多Dream비교과센터

- ASP(After School Program)는 1학년 학생이 참여하는 비교과 활동으로, 1인당 최소 1개의 동아리에 가입해 활동합니다. 학과 동아리와 중앙 동아리 등 약 400개의 동아리가 운영됩니다. 대학공통 필수 항목 'ASP(1,2학기)'가 이 활동입니다.
  출처: 순천향대 신문 「첫발을 내딛는 신입생들을 위해」 http://news.sch.ac.kr/news/articleView.html?idxno=488
- 多Dream비교과센터는 다면적졸업인증제 업무, 교내·외 비교과 활동의 수집·분류 및 전산관리 등을 담당합니다.
- 위치: 충남 아산시 순천향로 22, 향설생활관1 RC306호 / 전화: 041-530-1593
  출처: 순천향대학교 교육혁신원 多Dream비교과센터 https://inno.sch.ac.kr/sub/main_04_02

### 1-3. 안내받은 다른 문서 추가 (선택)
`UPLOAD_MORE` 를 체크하고 실행하면 파일 선택 창이 뜹니다. (PDF/TXT/MD/DOCX, HWP는 PDF로 변환해서)

In [ ]:
UPLOAD_MORE = False  #@param {type:"boolean"}
if UPLOAD_MORE:
    from google.colab import files
    for name, data in files.upload().items():
        open(f"docs/{name}", "wb").write(data)
        print("추가:", name)
import os; print("docs/:", os.listdir("docs"))

### 1-4. PDF 표 → 구조화 데이터 (`requirements.json`)
표 PDF는 텍스트로 추출하면 열이 섞여서 '어느 학번 몇 점'의 연결이 깨집니다.
그래서 표를 JSON으로 구조화해 **① 학과·학번별 검색용 청크**와 **② 졸업 판정 규칙**에 함께 사용합니다.

In [ ]:
%%writefile data/requirements.json
{
 "meta": {
  "title": "2026 학과별 다면적 졸업인증제 (SW융합대학)",
  "source": "2026_다면적졸업인증제_SW융합대학.pdf",
  "check_path": "차세대 종합정보시스템(https://st.sch.ac.kr) → 학부학사행정 → 졸업 → 개인별졸업진도관리 → 비교과일람표",
  "apply_path": "차세대 종합정보시스템(https://st.sch.ac.kr) → 학부학사행정 → 비교과과정 → 신청관리 → 비교과신청",
  "contact": "多Dream비교과센터 (향설 1관 RC306호)",
  "type_help": {
   "course": "교과목 이수 여부 (수강 과목 리스트에서 자동 인식, 또는 직접 체크)",
   "choose": "여러 과목 중 n개 이수",
   "check": "참여/이수 여부 체크",
   "count": "참여 횟수 × 회당 점수",
   "points": "취득 점수 직접 입력 (세부 항목 합산)",
   "topcit_rule": "TOPCIT 응시 횟수와 달성 수준으로 판정",
   "all_of": "여러 부분을 모두 이수해야 인정 (예: ASP 1·2학기)"
  },
  "structure": "구분은 대학공통(교과·비교과)과 학과기준(교과·졸업필수활동·비교과지정·비교과미지정) 두 단계로 나뉜다."
 },
 "common": [
  {
   "years": [
    2019,
    2022
   ],
   "items": [
    {
     "id": "c_human",
     "name": "인간사랑",
     "category": "소통하는 공감인재",
     "kind": "교과",
     "type": "course",
     "points": 5,
     "match": [
      "인간사랑"
     ]
    },
    {
     "id": "c_asp",
     "name": "ASP(1,2학기)",
     "category": "소통하는 공감인재",
     "kind": "비교과",
     "type": "all_of",
     "points": 5,
     "note": "1학기·2학기 모두 이수해야 5점",
     "parts": [
      "1학기",
      "2학기"
     ]
    },
    {
     "id": "c_selfcounsel",
     "name": "자기주도상담",
     "category": "소통하는 공감인재",
     "kind": "비교과",
     "type": "count",
     "points": 5,
     "per": 1,
     "unit": "회",
     "note": "회당 1점 · 5회 실시 시 5점 (학생 안내 기준, 안내문 PDF에는 배점 방식 미기재)"
    },
    {
     "id": "c_mm",
     "name": "자기개발과 전공탐색",
     "category": "융합하는 창의인재",
     "kind": "교과",
     "type": "course",
     "points": 5,
     "match": [
      "자기개발과전공탐색",
      "자기개발과전공탐색(M&M)",
      "M&M"
     ]
    },
    {
     "id": "c_gradact",
     "name": "졸업필수활동",
     "category": "융합하는 창의인재",
     "kind": "비교과",
     "type": "points",
     "points": 15,
     "subitems": [],
     "note": "세부 프로그램별 점수가 합산되는 항목 · 비교과일람표의 취득 점수 입력"
    },
    {
     "id": "c_eng",
     "name": "현대기본영어",
     "category": "공헌하는 글로벌인재",
     "kind": "교과",
     "type": "course",
     "points": 5,
     "match": [
      "현대기본영어"
     ]
    },
    {
     "id": "c_volunteer",
     "name": "봉사학습",
     "category": "공헌하는 글로벌인재",
     "kind": "교과",
     "type": "course",
     "points": 5,
     "match": [
      "봉사학습"
     ]
    }
   ]
  },
  {
   "years": [
    2023,
    2023
   ],
   "items": [
    {
     "id": "c_human",
     "name": "인간사랑",
     "category": "소통하는 공감인재",
     "kind": "교과",
     "type": "course",
     "points": 5,
     "match": [
      "인간사랑"
     ]
    },
    {
     "id": "c_asp",
     "name": "ASP(1,2학기)",
     "category": "소통하는 공감인재",
     "kind": "비교과",
     "type": "all_of",
     "points": 5,
     "note": "1학기·2학기 모두 이수해야 5점",
     "parts": [
      "1학기",
      "2학기"
     ]
    },
    {
     "id": "c_selfcounsel",
     "name": "자기주도상담",
     "category": "소통하는 공감인재",
     "kind": "비교과",
     "type": "count",
     "points": 5,
     "per": 1,
     "unit": "회",
     "note": "회당 1점 · 5회 실시 시 5점 (학생 안내 기준, 안내문 PDF에는 배점 방식 미기재)"
    },
    {
     "id": "c_mm",
     "name": "자기개발과 전공탐색(M&M)",
     "category": "융합하는 창의인재",
     "kind": "교과",
     "type": "course",
     "points": 5,
     "match": [
      "자기개발과전공탐색",
      "자기개발과전공탐색(M&M)",
      "M&M"
     ]
    },
    {
     "id": "c_gradguide",
     "name": "졸업종합지도",
     "category": "융합하는 창의인재",
     "kind": "비교과",
     "type": "points",
     "points": 15,
     "subitems": [],
     "note": "세부 프로그램별 점수가 합산되는 항목 · 비교과일람표의 취득 점수 입력"
    },
    {
     "id": "c_eng",
     "name": "현대기본영어",
     "category": "공헌하는 글로벌인재",
     "kind": "교과",
     "type": "course",
     "points": 5,
     "match": [
      "현대기본영어"
     ]
    },
    {
     "id": "c_volunteer",
     "name": "봉사학습",
     "category": "공헌하는 글로벌인재",
     "kind": "교과",
     "type": "course",
     "points": 5,
     "match": [
      "봉사학습"
     ]
    }
   ]
  },
  {
   "years": [
    2024,
    2026
   ],
   "items": [
    {
     "id": "c_human",
     "name": "인간사랑",
     "category": "소통하는 공감인재",
     "kind": "교과",
     "type": "course",
     "points": 5,
     "match": [
      "인간사랑"
     ]
    },
    {
     "id": "c_asp",
     "name": "ASP(1,2학기)",
     "category": "소통하는 공감인재",
     "kind": "비교과",
     "type": "all_of",
     "points": 5,
     "note": "1학기·2학기 모두 이수해야 5점",
     "parts": [
      "1학기",
      "2학기"
     ]
    },
    {
     "id": "c_selfcounsel",
     "name": "자기주도상담",
     "category": "소통하는 공감인재",
     "kind": "비교과",
     "type": "count",
     "points": 5,
     "per": 1,
     "unit": "회",
     "note": "회당 1점 · 5회 실시 시 5점 (학생 안내 기준, 안내문 PDF에는 배점 방식 미기재)"
    },
    {
     "id": "c_mm",
     "name": "자기개발과 전공탐색(M&M)",
     "category": "융합하는 창의인재",
     "kind": "교과",
     "type": "course",
     "points": 5,
     "match": [
      "자기개발과전공탐색",
      "자기개발과전공탐색(M&M)",
      "M&M"
     ]
    },
    {
     "id": "c_career",
     "name": "졸업및취·창업지도",
     "category": "융합하는 창의인재",
     "kind": "비교과",
     "type": "points",
     "points": 13,
     "subitems": [],
     "note": "세부 프로그램별 점수가 합산되는 항목 · 비교과일람표의 취득 점수 입력"
    },
    {
     "id": "c_ethics1",
     "name": "학습윤리-기본",
     "category": "융합하는 창의인재",
     "kind": "비교과",
     "type": "check",
     "points": 1
    },
    {
     "id": "c_ethics2",
     "name": "학습윤리-심화",
     "category": "융합하는 창의인재",
     "kind": "비교과",
     "type": "check",
     "points": 1
    },
    {
     "id": "c_eng",
     "name": "현대실용영어",
     "category": "공헌하는 글로벌인재",
     "kind": "교과",
     "type": "course",
     "points": 5,
     "match": [
      "현대실용영어"
     ]
    },
    {
     "id": "c_volunteer",
     "name": "봉사학습",
     "category": "공헌하는 글로벌인재",
     "kind": "교과",
     "type": "course",
     "points": 5,
     "match": [
      "봉사학습"
     ]
    }
   ]
  }
 ],
 "departments": {
  "컴퓨터소프트웨어공학과": {
   "aliases": [
    "컴소",
    "컴소공",
    "컴퓨터소프트웨어",
    "컴퓨터소프트웨어공학",
    "컴퓨터공학"
   ],
   "unit_rules": "전공상담(회당 1점), 학과활동(회당 1점), 학과MT참여(회당 2점), TOPCIT 응시(회당 1점)",
   "notes": [
    "2016~2021학번까지 전공영어1~4중 택3을 수강하지 못한 경우 전공영어기초, 전공영어응용, 예비창업설계를 수강시 대체 과목 인정"
   ],
   "tracks": [
    {
     "years": [
      2016,
      2020
     ],
     "total": 90,
     "subtotals": {
      "대학공통": 45,
      "교과": 25,
      "비교과지정": 14,
      "비교과미지정": 6
     },
     "items": [
      {
       "id": "counsel",
       "name": "전공상담",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 4
      },
      {
       "id": "free_comm",
       "name": "비교과자율활동 (*학과활동)",
       "category": "소통하는 공감인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 6,
       "subitems": [
        {
         "name": "학과활동",
         "mode": "count",
         "per": 1
        }
       ]
      },
      {
       "id": "capstone1",
       "name": "캡스톤디자인1",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "캡스톤디자인1"
       ]
      },
      {
       "id": "capstone2",
       "name": "캡스톤디자인2",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "캡스톤디자인2"
       ]
      },
      {
       "id": "symposium",
       "name": "학술제 발표",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 10
      },
      {
       "id": "major_eng",
       "name": "전공영어1~4 중 택3",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "choose",
       "n": 3,
       "points": 15,
       "options": [
        "전공영어1",
        "전공영어2",
        "전공영어3",
        "전공영어4"
       ],
       "substitutes": [
        "전공영어기초",
        "전공영어응용",
        "예비창업설계"
       ]
      }
     ]
    },
    {
     "years": [
      2021,
      2021
     ],
     "total": 90,
     "subtotals": {
      "대학공통": 45,
      "교과": 25,
      "비교과지정": 15,
      "비교과미지정": 5
     },
     "items": [
      {
       "id": "counsel",
       "name": "전공상담",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 4
      },
      {
       "id": "dept_meeting",
       "name": "학과활동 (*4학년 간담회 참석)",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 1
      },
      {
       "id": "free_comm",
       "name": "비교과자율활동 (*학과활동)",
       "category": "소통하는 공감인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": [
        {
         "name": "학과활동",
         "mode": "count",
         "per": 1
        }
       ]
      },
      {
       "id": "capstone1",
       "name": "캡스톤디자인1",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "캡스톤디자인1"
       ]
      },
      {
       "id": "capstone2",
       "name": "캡스톤디자인2",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "캡스톤디자인2"
       ]
      },
      {
       "id": "symposium",
       "name": "학술제 발표",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 10
      },
      {
       "id": "major_eng",
       "name": "전공영어1~4 중 택3",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "choose",
       "n": 3,
       "points": 15,
       "options": [
        "전공영어1",
        "전공영어2",
        "전공영어3",
        "전공영어4"
       ],
       "substitutes": [
        "전공영어기초",
        "전공영어응용",
        "예비창업설계"
       ]
      }
     ]
    },
    {
     "years": [
      2022,
      2024
     ],
     "total": 104,
     "subtotals": {
      "대학공통": 45,
      "교과": 25,
      "비교과지정": 24,
      "비교과미지정": 10
     },
     "items": [
      {
       "id": "counsel",
       "name": "전공상담",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 4
      },
      {
       "id": "capstone",
       "name": "캡스톤디자인",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "캡스톤디자인"
       ]
      },
      {
       "id": "ind_capstone",
       "name": "산학캡스톤디자인",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "산학캡스톤디자인"
       ]
      },
      {
       "id": "symposium",
       "name": "학술제발표",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 10
      },
      {
       "id": "coding",
       "name": "코딩역량인증제(Level2 이상)",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "aisw",
       "name": "AI·SW전공역량졸업인증제",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동 (*맛있는SW시리즈, *TOPCIT 응시)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 10,
       "subitems": [
        {
         "name": "TOPCIT 응시",
         "mode": "count",
         "per": 1
        },
        {
         "name": "맛있는SW시리즈 (취득 점수 입력)",
         "mode": "number"
        }
       ]
      },
      {
       "id": "eng_basic",
       "name": "전공영어기초",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "전공영어기초"
       ]
      },
      {
       "id": "eng_apply",
       "name": "전공영어응용",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "전공영어응용"
       ]
      },
      {
       "id": "startup",
       "name": "예비창업설계",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "예비창업설계"
       ]
      }
     ]
    },
    {
     "years": [
      2025,
      2026
     ],
     "total": 104,
     "subtotals": {
      "대학공통": 45,
      "교과": 25,
      "비교과지정": 24,
      "비교과미지정": 10
     },
     "items": [
      {
       "id": "counsel",
       "name": "전공상담",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 4
      },
      {
       "id": "mt",
       "name": "학과 MT참여",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 2,
       "unit": "회",
       "points": 4
      },
      {
       "id": "capstone",
       "name": "캡스톤디자인",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "캡스톤디자인"
       ]
      },
      {
       "id": "ind_capstone",
       "name": "산학캡스톤디자인",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "산학캡스톤디자인"
       ]
      },
      {
       "id": "symposium",
       "name": "학술제발표",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 6
      },
      {
       "id": "coding",
       "name": "코딩역량인증제(Level2 이상)",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "aisw",
       "name": "AI·SW전공역량졸업인증제",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동 (*맛있는SW시리즈, *TOPCIT 응시)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 10,
       "subitems": [
        {
         "name": "TOPCIT 응시",
         "mode": "count",
         "per": 1
        },
        {
         "name": "맛있는SW시리즈 (취득 점수 입력)",
         "mode": "number"
        }
       ]
      },
      {
       "id": "eng_basic",
       "name": "전공영어기초",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "전공영어기초"
       ]
      },
      {
       "id": "eng_apply",
       "name": "전공영어응용",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "전공영어응용"
       ]
      },
      {
       "id": "startup",
       "name": "예비창업설계",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "예비창업설계"
       ]
      }
     ]
    }
   ]
  },
  "정보보호학과": {
   "aliases": [
    "정보보호",
    "정보보안",
    "정보보호학"
   ],
   "unit_rules": "학과활동(회당 1점), 학과학술동아리(학기당 3점), TOPCIT 응시(회당 1점)",
   "notes": [
    "2016~2021학번까지 전공영어1, 전공영어2를 수강하지 못했을 경우 대체과목으로 '전공영어1', '전공영어2', '전공영어3', '전공영어기초2', '전공영어응용' 중 택2"
   ],
   "tracks": [
    {
     "years": [
      2016,
      2021
     ],
     "total": 85,
     "subtotals": {
      "대학공통": 45,
      "교과": 15,
      "비교과지정": 20,
      "비교과미지정": 5
     },
     "items": [
      {
       "id": "dept_act",
       "name": "학과활동",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 10
      },
      {
       "id": "capstone34",
       "name": "캡스톤디자인3~4 택1",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "choose",
       "n": 1,
       "points": 5,
       "options": [
        "캡스톤디자인3",
        "캡스톤디자인4"
       ]
      },
      {
       "id": "symposium",
       "name": "(단과대)학술제발표",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 10
      },
      {
       "id": "major_eng",
       "name": "전공영어기초 + 전공영어응용 (대체: 전공영어1·2·3, 전공영어기초2, 전공영어응용 중 택2)",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "choose",
       "n": 2,
       "points": 10,
       "options": [
        "전공영어기초",
        "전공영어응용",
        "전공영어1",
        "전공영어2",
        "전공영어3",
        "전공영어기초2"
       ]
      },
      {
       "id": "free_global",
       "name": "비교과자율활동",
       "category": "공헌하는 글로벌인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": []
      }
     ]
    },
    {
     "years": [
      2022,
      2022
     ],
     "total": 110,
     "subtotals": {
      "대학공통": 45,
      "교과": 25,
      "비교과지정": 30,
      "비교과미지정": 10
     },
     "items": [
      {
       "id": "dept_act",
       "name": "학과활동",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 10
      },
      {
       "id": "capstone_or_thesis",
       "name": "산학캡스톤디자인2 혹은 졸업논문프로젝트2 택1",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "choose",
       "n": 1,
       "points": 5,
       "options": [
        "산학캡스톤디자인2",
        "졸업논문프로젝트2"
       ]
      },
      {
       "id": "cprog",
       "name": "C프로그래밍",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "C프로그래밍"
       ]
      },
      {
       "id": "symposium",
       "name": "(단과대)학술제발표",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 10
      },
      {
       "id": "coding",
       "name": "코딩역량인증제",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "aisw",
       "name": "AI·SW전공역량졸업인증제",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동 (*맛있는SW, *학과학술동아리)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 10,
       "subitems": [
        {
         "name": "맛있는SW",
         "mode": "check",
         "points": 5
        },
        {
         "name": "학과학술동아리 (학기 수)",
         "mode": "count",
         "per": 3
        }
       ]
      },
      {
       "id": "eng_basic",
       "name": "전공영어기초",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "전공영어기초"
       ]
      },
      {
       "id": "eng_apply",
       "name": "전공영어응용",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "전공영어응용"
       ]
      },
      {
       "id": "startup",
       "name": "예비창업설계",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "예비창업설계"
       ]
      }
     ]
    },
    {
     "years": [
      2023,
      2026
     ],
     "total": 110,
     "subtotals": {
      "대학공통": 45,
      "교과": 25,
      "비교과지정": 30,
      "비교과미지정": 10
     },
     "items": [
      {
       "id": "dept_act",
       "name": "학과활동",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 10
      },
      {
       "id": "capstone_or_thesis",
       "name": "산학캡스톤디자인2 혹은 졸업논문프로젝트2 택1",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "choose",
       "n": 1,
       "points": 5,
       "options": [
        "산학캡스톤디자인2",
        "졸업논문프로젝트2"
       ]
      },
      {
       "id": "cprog",
       "name": "C프로그래밍",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "C프로그래밍"
       ]
      },
      {
       "id": "symposium",
       "name": "(단과대)학술제발표",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "coding",
       "name": "코딩역량인증제",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "aisw",
       "name": "AI·SW전공역량졸업인증제",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "topcit",
       "name": "TOPCIT 응시",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 5
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동 (*맛있는SW, *학과학술동아리)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 10,
       "subitems": [
        {
         "name": "맛있는SW",
         "mode": "check",
         "points": 5
        },
        {
         "name": "학과학술동아리 (학기 수)",
         "mode": "count",
         "per": 3
        }
       ]
      },
      {
       "id": "eng_basic",
       "name": "전공영어기초",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "전공영어기초"
       ]
      },
      {
       "id": "eng_apply",
       "name": "전공영어응용",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "전공영어응용"
       ]
      },
      {
       "id": "startup",
       "name": "예비창업설계",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "예비창업설계"
       ]
      }
     ]
    }
   ]
  },
  "의료IT공학과": {
   "aliases": [
    "의료IT",
    "의료아이티",
    "의료IT공학"
   ],
   "unit_rules": "MT참여(회당 1점)",
   "notes": [],
   "tracks": [
    {
     "years": [
      2021,
      2021
     ],
     "total": 80,
     "subtotals": {
      "대학공통": 45,
      "교과": 15,
      "비교과지정": 6,
      "비교과미지정": 14
     },
     "items": [
      {
       "id": "mt",
       "name": "학과 MT",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 1
      },
      {
       "id": "free_comm",
       "name": "비교과자율활동(공감)",
       "category": "소통하는 공감인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 7,
       "subitems": []
      },
      {
       "id": "medterm",
       "name": "의학용어",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "의학용어"
       ]
      },
      {
       "id": "presentation",
       "name": "프레젠테이션테크닉",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "프레젠테이션테크닉"
       ]
      },
      {
       "id": "symposium_join",
       "name": "학술제참여",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 1
      },
      {
       "id": "expo",
       "name": "전공관련국제박람회",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 2
      },
      {
       "id": "career_prog",
       "name": "진로(취업)지도관련 학내 프로그램 참여",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 2
      },
      {
       "id": "app_eng",
       "name": "현대응용영어",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "현대응용영어"
       ]
      },
      {
       "id": "free_global",
       "name": "비교과자율활동(글로벌)",
       "category": "공헌하는 글로벌인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 7,
       "subitems": []
      }
     ]
    },
    {
     "years": [
      2022,
      2023
     ],
     "total": 88,
     "subtotals": {
      "대학공통": 45,
      "교과": 15,
      "비교과지정": 13,
      "비교과미지정": 15
     },
     "items": [
      {
       "id": "mt",
       "name": "MT참여 2회",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 2
      },
      {
       "id": "free_comm",
       "name": "비교과자율활동(공감)",
       "category": "소통하는 공감인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": []
      },
      {
       "id": "medterm",
       "name": "의학용어",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "의학용어"
       ]
      },
      {
       "id": "ct",
       "name": "컴퓨팅적사고와문제해결",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "컴퓨팅적사고와문제해결"
       ]
      },
      {
       "id": "symposium_join",
       "name": "학술제 참여",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 1
      },
      {
       "id": "coding",
       "name": "코딩역량인증 평가 Level2 이상",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "aisw",
       "name": "AI·SW전공역량 졸업인증제 200점 이상",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동(창의) (*국제박람회, *학술제 작품발표, *맛있는SW)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": [
        {
         "name": "전공관련 국제박람회",
         "mode": "check",
         "points": 2
        },
        {
         "name": "학술제 작품발표(단과대)",
         "mode": "check",
         "points": 2
        },
        {
         "name": "맛있는SW",
         "mode": "check",
         "points": 5
        }
       ]
      },
      {
       "id": "app_eng",
       "name": "현대응용영어",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "현대응용영어"
       ]
      },
      {
       "id": "free_global",
       "name": "비교과자율활동(글로벌)",
       "category": "공헌하는 글로벌인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": []
      }
     ]
    },
    {
     "years": [
      2024,
      2026
     ],
     "total": 89,
     "subtotals": {
      "대학공통": 45,
      "교과": 15,
      "비교과지정": 14,
      "비교과미지정": 15
     },
     "items": [
      {
       "id": "mt",
       "name": "MT참여 2회",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 2
      },
      {
       "id": "free_comm",
       "name": "비교과자율활동(공감)",
       "category": "소통하는 공감인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": []
      },
      {
       "id": "medterm",
       "name": "의학용어",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "의학용어"
       ]
      },
      {
       "id": "coding_course",
       "name": "컴퓨터코딩과문제해결",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "컴퓨터코딩과문제해결"
       ]
      },
      {
       "id": "symposium_join",
       "name": "학술제 참여",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 1
      },
      {
       "id": "coding",
       "name": "코딩역량인증 평가 Level2 이상",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "aisw",
       "name": "AI·SW전공역량 졸업인증제 200점 이상",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "topcit_lv1",
       "name": "TOPCIT 수준1 달성",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 1
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동(창의) (*국제박람회, *학술제 작품발표, *맛있는SW)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": [
        {
         "name": "전공관련 국제박람회",
         "mode": "check",
         "points": 2
        },
        {
         "name": "학술제 작품발표(단과대)",
         "mode": "check",
         "points": 2
        },
        {
         "name": "맛있는SW",
         "mode": "check",
         "points": 5
        }
       ]
      },
      {
       "id": "app_eng",
       "name": "현대응용영어",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "현대응용영어"
       ]
      },
      {
       "id": "free_global",
       "name": "비교과자율활동(글로벌)",
       "category": "공헌하는 글로벌인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": []
      }
     ]
    }
   ]
  },
  "AI·빅데이터학과": {
   "aliases": [
    "AI빅데이터",
    "ai빅데이터",
    "AI·빅데이터",
    "에이아이빅데이터",
    "빅데이터",
    "AI빅데",
    "인공지능빅데이터"
   ],
   "unit_rules": "학과활동(회당 1점), TOPCIT 응시(회당 1점), 경진대회 참가(3~10점), 경진대회 수상(10~30점), 권장 자격증 취득(6점): 데이터아키텍처 전문가 / SQL전문가 / SQL개발자 / 데이터분석전문가 / 데이터분석준전문가 / 빅데이터분석기사",
   "notes": [
    "권장수강학기: 예비창업설계(2학년 2학기)",
    "권장 자격증(6점): 데이터아키텍처 전문가(DAP), SQL전문가(SQLP), SQL개발자(SQLD), 데이터분석전문가(ADP), 데이터분석준전문가(ADsP), 빅데이터분석기사"
   ],
   "tracks": [
    {
     "years": [
      2019,
      2021
     ],
     "total": 80,
     "subtotals": {
      "대학공통": 45,
      "교과": 5,
      "비교과지정": 11,
      "비교과미지정": 19
     },
     "items": [
      {
       "id": "dept_act",
       "name": "학과활동",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 5
      },
      {
       "id": "free_comm",
       "name": "비교과자율활동(공감)",
       "category": "소통하는 공감인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": []
      },
      {
       "id": "cert_contest",
       "name": "자격증/경진대회참가/경진대회수상",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "points",
       "points": 6,
       "subitems": [
        {
         "name": "데이터아키텍처 전문가(DAP) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "SQL전문가(SQLP) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "SQL개발자(SQLD) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "데이터분석전문가(ADP) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "데이터분석준전문가(ADsP) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "빅데이터분석기사 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "경진대회 참가 (3~10점, 인정 점수 입력)",
         "mode": "number"
        },
        {
         "name": "경진대회 수상 (10~30점, 인정 점수 입력)",
         "mode": "number"
        }
       ]
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동(창의)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 4,
       "subitems": []
      },
      {
       "id": "core_lang",
       "name": "중핵교양-국제사회언어영역 과목 택1",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "check",
       "points": 5,
       "note": "중핵교양 국제사회언어영역에 해당하는 과목 1개 이수 시 체크"
      },
      {
       "id": "free_global",
       "name": "비교과자율활동(글로벌)",
       "category": "공헌하는 글로벌인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 10,
       "subitems": []
      }
     ]
    },
    {
     "years": [
      2022,
      2022
     ],
     "total": 95,
     "subtotals": {
      "대학공통": 30,
      "교과": 10,
      "졸업필수활동": 15,
      "비교과지정": 21,
      "비교과미지정": 19
     },
     "items": [
      {
       "id": "dept_act",
       "name": "학과활동",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 5
      },
      {
       "id": "cert_contest",
       "name": "자격증/경진대회참가/경진대회수상",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "points",
       "points": 6,
       "subitems": [
        {
         "name": "데이터아키텍처 전문가(DAP) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "SQL전문가(SQLP) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "SQL개발자(SQLD) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "데이터분석전문가(ADP) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "데이터분석준전문가(ADsP) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "빅데이터분석기사 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "경진대회 참가 (3~10점, 인정 점수 입력)",
         "mode": "number"
        },
        {
         "name": "경진대회 수상 (10~30점, 인정 점수 입력)",
         "mode": "number"
        }
       ]
      },
      {
       "id": "coding",
       "name": "코딩역량인증제(Lv2 이상)",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "aisw",
       "name": "AI·SW전공역량졸업인증제",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동(창의)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 9,
       "subitems": [],
       "note": "비교과일람표(학생 확인) 기준 · 안내문 PDF 표와 다름 (PDF: 공감 5·창의 4·글로벌 10, 일람표: 공감 없음·창의 9·글로벌 10)"
      },
      {
       "id": "startup",
       "name": "예비창업설계",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "예비창업설계"
       ],
       "note": "권장수강학기: 2학년 2학기 · 인재상: 글로벌 (비교과일람표 기준, PDF 표에는 창의로 표기)"
      },
      {
       "id": "core_lang",
       "name": "중핵교양-국제사회언어영역 과목 택1",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "check",
       "points": 5,
       "note": "중핵교양 국제사회언어영역에 해당하는 과목 1개 이수 시 체크"
      },
      {
       "id": "free_global",
       "name": "비교과자율활동(글로벌)",
       "category": "공헌하는 글로벌인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 10,
       "subitems": []
      }
     ],
     "common_overrides": {
      "c_gradact": {
       "group": "학과기준",
       "kind": "졸업필수활동",
       "note": "비교과일람표(학생 확인) 기준: 학과기준의 졸업필수활동으로 분류 (안내문 PDF에는 대학공통으로 표기)"
      }
     }
    },
    {
     "years": [
      2023,
      2026
     ],
     "total": 95,
     "subtotals": {
      "대학공통": 45,
      "교과": 5,
      "비교과지정": 26,
      "비교과미지정": 19
     },
     "items": [
      {
       "id": "dept_act",
       "name": "학과활동",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 5
      },
      {
       "id": "free_comm",
       "name": "비교과자율활동(공감)",
       "category": "소통하는 공감인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": []
      },
      {
       "id": "startup",
       "name": "예비창업설계",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "예비창업설계"
       ],
       "note": "권장수강학기: 2학년 2학기"
      },
      {
       "id": "cert_contest",
       "name": "자격증/경진대회참가/경진대회수상",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "points",
       "points": 6,
       "subitems": [
        {
         "name": "데이터아키텍처 전문가(DAP) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "SQL전문가(SQLP) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "SQL개발자(SQLD) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "데이터분석전문가(ADP) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "데이터분석준전문가(ADsP) 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "빅데이터분석기사 취득",
         "mode": "check",
         "points": 6
        },
        {
         "name": "경진대회 참가 (3~10점, 인정 점수 입력)",
         "mode": "number"
        },
        {
         "name": "경진대회 수상 (10~30점, 인정 점수 입력)",
         "mode": "number"
        }
       ]
      },
      {
       "id": "coding",
       "name": "코딩역량인증제(Lv2 이상)",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "aisw",
       "name": "AI·SW전공역량졸업인증제",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "topcit",
       "name": "TOPCIT 응시",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 5
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동(창의)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 4,
       "subitems": []
      },
      {
       "id": "free_global",
       "name": "비교과자율활동(글로벌)",
       "category": "공헌하는 글로벌인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 10,
       "subitems": []
      }
     ]
    }
   ]
  },
  "사물인터넷학과": {
   "aliases": [
    "사물인터넷",
    "IoT",
    "iot",
    "사인"
   ],
   "unit_rules": "지도교수상담·졸업상담(회당 1점), 학과활동(회당 1점), TOPCIT 응시(회당 1점)",
   "notes": [
    "권장수강학기: 예비창업설계(2학년 1학기)"
   ],
   "tracks": [
    {
     "years": [
      2017,
      2018
     ],
     "total": 80,
     "subtotals": {
      "대학공통": 45,
      "교과": 10,
      "비교과지정": 10,
      "비교과미지정": 15
     },
     "items": [
      {
       "id": "advisor",
       "name": "지도교수상담",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 3
      },
      {
       "id": "gradcounsel",
       "name": "졸업상담",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 2
      },
      {
       "id": "dept_act",
       "name": "학과활동",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 5
      },
      {
       "id": "capstone2plus",
       "name": "캡스톤디자인 2과목 이상",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "choose",
       "n": 2,
       "points": 10,
       "pattern": "캡스톤디자인"
      },
      {
       "id": "free_global",
       "name": "비교과자율활동(글로벌) (*토익500↑, *OPIc M1↑, *토익스피킹100↑)",
       "category": "공헌하는 글로벌인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 15,
       "subitems": [
        {
         "name": "토익 500점 이상",
         "mode": "check",
         "points": 5
        },
        {
         "name": "OPIc M1 이상",
         "mode": "check",
         "points": 5
        },
        {
         "name": "토익스피킹 100점 이상",
         "mode": "check",
         "points": 5
        }
       ]
      }
     ]
    },
    {
     "years": [
      2019,
      2021
     ],
     "total": 80,
     "subtotals": {
      "대학공통": 45,
      "교과": 15,
      "비교과지정": 10,
      "비교과미지정": 10
     },
     "items": [
      {
       "id": "advisor",
       "name": "지도교수상담",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 3
      },
      {
       "id": "gradcounsel",
       "name": "졸업상담",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 2
      },
      {
       "id": "dept_act",
       "name": "학과활동",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 5
      },
      {
       "id": "capstone2plus",
       "name": "캡스톤디자인 2과목 이상",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "choose",
       "n": 2,
       "points": 10,
       "pattern": "캡스톤디자인"
      },
      {
       "id": "core_eng",
       "name": "중핵교양 국제사회언어영역 '영어' 교과목 이수",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "check",
       "points": 5
      },
      {
       "id": "free_global",
       "name": "비교과자율활동(글로벌) (*토익500↑, *OPIc M1↑, *토익스피킹100↑)",
       "category": "공헌하는 글로벌인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 10,
       "subitems": [
        {
         "name": "토익 500점 이상",
         "mode": "check",
         "points": 5
        },
        {
         "name": "OPIc M1 이상",
         "mode": "check",
         "points": 5
        },
        {
         "name": "토익스피킹 100점 이상",
         "mode": "check",
         "points": 5
        }
       ]
      }
     ]
    },
    {
     "years": [
      2022,
      2022
     ],
     "total": 80,
     "subtotals": {
      "대학공통": 45,
      "교과": 5,
      "비교과지정": 20,
      "비교과미지정": 10
     },
     "items": [
      {
       "id": "advisor",
       "name": "지도교수상담",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 3
      },
      {
       "id": "gradcounsel",
       "name": "졸업상담",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 2
      },
      {
       "id": "dept_act",
       "name": "학과활동",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 5
      },
      {
       "id": "startup",
       "name": "예비창업설계",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "예비창업설계"
       ]
      },
      {
       "id": "coding",
       "name": "코딩역량인증제(Lv2 이상)",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "aisw",
       "name": "AI·SW전공역량 졸업인증제",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동(창의) (*맛있는SW)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": [
        {
         "name": "맛있는SW",
         "mode": "check",
         "points": 5
        }
       ]
      },
      {
       "id": "free_global",
       "name": "비교과자율활동(글로벌) (*토익500↑, *OPIc M1↑, *토익스피킹100↑)",
       "category": "공헌하는 글로벌인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": [
        {
         "name": "토익 500점 이상",
         "mode": "check",
         "points": 5
        },
        {
         "name": "OPIc M1 이상",
         "mode": "check",
         "points": 5
        },
        {
         "name": "토익스피킹 100점 이상",
         "mode": "check",
         "points": 5
        }
       ]
      }
     ]
    },
    {
     "years": [
      2023,
      2023
     ],
     "total": 81,
     "subtotals": {
      "대학공통": 45,
      "교과": 5,
      "비교과지정": 21,
      "비교과미지정": 10
     },
     "items": [
      {
       "id": "advisor",
       "name": "지도교수상담 3회",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 3
      },
      {
       "id": "gradcounsel",
       "name": "졸업상담 2회",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 2
      },
      {
       "id": "dept_act",
       "name": "학과활동 5회",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 5
      },
      {
       "id": "startup",
       "name": "예비창업설계",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "예비창업설계"
       ]
      },
      {
       "id": "topcit",
       "name": "TOPCIT 응시",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 1
      },
      {
       "id": "coding",
       "name": "코딩역량인증제(Lv2 이상)",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "aisw",
       "name": "AI·SW전공역량 졸업인증제",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동(창의) (*맛있는SW)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": [
        {
         "name": "맛있는SW",
         "mode": "check",
         "points": 5
        }
       ]
      },
      {
       "id": "free_global",
       "name": "비교과자율활동(글로벌) (*공인어학점수)",
       "category": "공헌하는 글로벌인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": [
        {
         "name": "공인어학점수 (인정 점수 입력)",
         "mode": "number"
        }
       ]
      }
     ]
    },
    {
     "years": [
      2024,
      2024
     ],
     "total": 81,
     "subtotals": {
      "대학공통": 45,
      "교과": 5,
      "비교과지정": 18,
      "비교과미지정": 13
     },
     "items": [
      {
       "id": "advisor",
       "name": "지도교수상담 2회",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 2
      },
      {
       "id": "dept_act",
       "name": "학과활동 5회",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 5
      },
      {
       "id": "startup",
       "name": "예비창업설계",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "예비창업설계"
       ]
      },
      {
       "id": "topcit",
       "name": "TOPCIT 응시",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 1
      },
      {
       "id": "coding",
       "name": "코딩역량인증제(Lv2 이상)",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "aisw",
       "name": "AI·SW전공역량 졸업인증제",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동(창의) (*맛있는SW, *정보처리기사, *ADsP, *SQLD)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 8,
       "subitems": [
        {
         "name": "맛있는SW",
         "mode": "check",
         "points": 5
        },
        {
         "name": "정보처리기사 응시",
         "mode": "check",
         "points": 2
        },
        {
         "name": "정보처리기사 취득",
         "mode": "check",
         "points": 10
        },
        {
         "name": "ADsP 취득",
         "mode": "check",
         "points": 5
        },
        {
         "name": "SQLD 취득",
         "mode": "check",
         "points": 5
        }
       ]
      },
      {
       "id": "free_global",
       "name": "비교과자율활동(글로벌) (*공인어학점수)",
       "category": "공헌하는 글로벌인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": [
        {
         "name": "공인어학점수 (인정 점수 입력)",
         "mode": "number"
        }
       ]
      }
     ]
    },
    {
     "years": [
      2025,
      2026
     ],
     "total": 84,
     "subtotals": {
      "대학공통": 45,
      "교과": 5,
      "비교과지정": 21,
      "비교과미지정": 13
     },
     "items": [
      {
       "id": "career_counsel",
       "name": "진로 및 취업상담 2회",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 2
      },
      {
       "id": "dept_act",
       "name": "학과활동 5회",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 5
      },
      {
       "id": "startup",
       "name": "예비창업설계",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "예비창업설계"
       ]
      },
      {
       "id": "topcit",
       "name": "TOPCIT 응시",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 4
      },
      {
       "id": "coding",
       "name": "코딩역량인증제(Lv2 이상)",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "aisw",
       "name": "AI·SW전공역량 졸업인증제",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동(창의) (*맛있는SW, *정보처리기사, *ADsP, *SQLD)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 8,
       "subitems": [
        {
         "name": "맛있는SW",
         "mode": "check",
         "points": 5
        },
        {
         "name": "정보처리기사 응시",
         "mode": "check",
         "points": 2
        },
        {
         "name": "정보처리기사 취득",
         "mode": "check",
         "points": 10
        },
        {
         "name": "ADsP 취득",
         "mode": "check",
         "points": 5
        },
        {
         "name": "SQLD 취득",
         "mode": "check",
         "points": 5
        }
       ]
      },
      {
       "id": "free_global",
       "name": "비교과자율활동(글로벌) (*공인어학점수)",
       "category": "공헌하는 글로벌인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": [
        {
         "name": "공인어학점수 (인정 점수 입력)",
         "mode": "number"
        }
       ]
      }
     ]
    }
   ]
  },
  "메타버스&게임학과": {
   "aliases": [
    "메타버스",
    "메타버스게임",
    "메타버스&게임",
    "게임학과",
    "메게"
   ],
   "unit_rules": "전공상담(회당 1점), 학과활동(회당 1점), TOPCIT 수준별 점수 부여(창의영역: 수준1 입문형 0~149 → 1점, 수준2 도전형 150~399 → 2점, 수준3 능숙형 400~649 → 4점, 수준4 문제해결형 650~849 → 5점, 수준5 창의융합형 850~1000 → 7점)",
   "notes": [
    "TOPCIT(5) 세부 배점 기준: 1) 응시 5회, 2) 응시 3회 + TOPCIT Lv2 달성, 3) 응시 1회 + TOPCIT Lv3 달성, 4) TOPCIT Lv4 달성 중 하나 충족"
   ],
   "tracks": [
    {
     "years": [
      2023,
      2025
     ],
     "total": 110,
     "subtotals": {
      "대학공통": 45,
      "교과": 25,
      "비교과지정": 30,
      "비교과미지정": 10
     },
     "items": [
      {
       "id": "counsel",
       "name": "전공상담",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 5
      },
      {
       "id": "free_comm",
       "name": "비교과자율활동(공감) (*학과활동)",
       "category": "소통하는 공감인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": [
        {
         "name": "학과활동",
         "mode": "count",
         "per": 1
        }
       ]
      },
      {
       "id": "ind_capstone",
       "name": "산학캡스톤디자인",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "산학캡스톤디자인"
       ]
      },
      {
       "id": "thesis",
       "name": "졸업논문프로젝트",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "졸업논문프로젝트"
       ]
      },
      {
       "id": "symposium",
       "name": "(단과대)학술제발표",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 10
      },
      {
       "id": "coding",
       "name": "코딩역량인증제 Lv2 이상",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "aisw",
       "name": "AI·SW전공역량졸업인증제",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "topcit",
       "name": "TOPCIT",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "topcit_rule",
       "points": 5
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동(창의) (*추천: 맛있는SW)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": [
        {
         "name": "맛있는SW",
         "mode": "check",
         "points": 5
        },
        {
         "name": "TOPCIT 수준별 점수",
         "mode": "select",
         "options": {
          "응시 안 함": 0,
          "수준1(입문형) 0~149": 1,
          "수준2(도전형) 150~399": 2,
          "수준3(능숙형) 400~649": 4,
          "수준4(문제해결형) 650~849": 5,
          "수준5(창의융합형) 850~1000": 7
         }
        }
       ]
      },
      {
       "id": "eng_basic",
       "name": "전공영어기초",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "전공영어기초"
       ]
      },
      {
       "id": "eng_apply",
       "name": "전공영어응용",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "전공영어응용"
       ]
      },
      {
       "id": "startup",
       "name": "예비창업설계",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "예비창업설계"
       ]
      }
     ]
    },
    {
     "years": [
      2026,
      2026
     ],
     "total": 110,
     "subtotals": {
      "대학공통": 45,
      "교과": 25,
      "비교과지정": 25,
      "비교과미지정": 15
     },
     "items": [
      {
       "id": "counsel",
       "name": "전공상담",
       "category": "소통하는 공감인재",
       "kind": "비교과지정",
       "type": "count",
       "per": 1,
       "unit": "회",
       "points": 5
      },
      {
       "id": "free_comm",
       "name": "비교과자율활동(공감) (*학과활동)",
       "category": "소통하는 공감인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 5,
       "subitems": [
        {
         "name": "학과활동",
         "mode": "count",
         "per": 1
        }
       ]
      },
      {
       "id": "ind_capstone",
       "name": "산학캡스톤디자인",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "산학캡스톤디자인"
       ]
      },
      {
       "id": "thesis",
       "name": "졸업논문프로젝트",
       "category": "융합하는 창의인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "졸업논문프로젝트"
       ]
      },
      {
       "id": "symposium",
       "name": "(단과대)학술제발표",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 10
      },
      {
       "id": "coding",
       "name": "코딩역량인증제(Lv2 이상)",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "aisw",
       "name": "AI·SW전공역량졸업인증제",
       "category": "융합하는 창의인재",
       "kind": "비교과지정",
       "type": "check",
       "points": 5
      },
      {
       "id": "free_creative",
       "name": "비교과자율활동(창의) (*추천: 맛있는SW시리즈, TOPCIT응시)",
       "category": "융합하는 창의인재",
       "kind": "비교과미지정",
       "type": "points",
       "points": 10,
       "subitems": [
        {
         "name": "맛있는SW시리즈",
         "mode": "check",
         "points": 5
        },
        {
         "name": "TOPCIT 응시",
         "mode": "check",
         "points": 5
        },
        {
         "name": "TOPCIT 수준별 점수",
         "mode": "select",
         "options": {
          "응시 안 함": 0,
          "수준1(입문형) 0~149": 1,
          "수준2(도전형) 150~399": 2,
          "수준3(능숙형) 400~649": 4,
          "수준4(문제해결형) 650~849": 5,
          "수준5(창의융합형) 850~1000": 7
         }
        }
       ]
      },
      {
       "id": "eng_basic",
       "name": "전공영어기초",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "전공영어기초"
       ]
      },
      {
       "id": "eng_apply",
       "name": "전공영어응용",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "전공영어응용"
       ]
      },
      {
       "id": "startup",
       "name": "예비창업설계",
       "category": "공헌하는 글로벌인재",
       "kind": "교과",
       "type": "course",
       "points": 5,
       "match": [
        "예비창업설계"
       ]
      }
     ]
    }
   ]
  }
 }
}

## 2. RAG 모듈
| 파일 | 역할 |
|---|---|
| `rag/checker.py` | 졸업요건 **규칙 엔진** (점수 계산은 LLM이 아니라 코드가 → 계산 실수 방지) |
| `rag/ingest.py` | 문서 로딩 → 청크 분할(700자/120자 겹침) + 구조화 청크 생성 |
| `rag/retriever.py` | 문자 n-gram TF-IDF + OpenAI 임베딩 하이브리드 검색, 학과/학번 메타데이터 부스팅 |
| `rag/llm.py` | 시스템 프롬프트, **tool calling** 루프(search_documents / get_requirements / check_graduation) |

In [ ]:
open('rag/__init__.py', 'w').close()

In [ ]:
%%writefile rag/checker.py
"""
다면적 졸업인증제 판정기 (규칙 기반)

LLM이 점수를 '추측'하지 않도록, 표의 내용은 data/requirements.json 으로 구조화하고
판정은 이 모듈이 결정적으로 계산한다. LLM은 결과를 설명하는 역할만 한다.

입력(activities) 형식 — 항목 type 별:
  course       : True/False  (수강 리스트에 없어도 이수했으면 True)
  check        : True/False
  all_of       : ["1학기", "2학기"] 처럼 이수한 부분 리스트 (또는 True = 전부)
  count        : 정수 (참여 횟수)
  choose       : ["과목명", ...]  (수강 리스트 외에 추가로 인정할 과목)
  points       : 숫자(총 인정 점수)  또는  {"subitems": {"세부항목명": 값}, "other": 숫자}
                 (세부항목 값: check=True, count=횟수, number=점수, select=선택지 이름)
  topcit_rule  : {"attempts": 응시횟수, "level": 최고 달성 수준(0~5)}
"""
from __future__ import annotations

import copy
import json
import re
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any

DATA_PATH = Path(__file__).resolve().parent.parent / "data" / "requirements.json"
GROUPS = ["대학공통", "학과기준"]
SECTIONS = [("대학공통", "교과"), ("대학공통", "비교과"),
            ("학과기준", "교과"), ("학과기준", "졸업필수활동"), ("학과기준", "비교과지정"), ("학과기준", "비교과미지정")]
DEPT_KINDS = ["교과", "졸업필수활동", "비교과지정", "비교과미지정"]


def section_label(group: str, kind: str) -> str:
    return f"{group}·{kind}"
CATEGORIES = ["소통하는 공감인재", "융합하는 창의인재", "공헌하는 글로벌인재"]


# ----------------------------------------------------------------------------
# 데이터 로딩 / 조회
# ----------------------------------------------------------------------------
def load_requirements(path: Path | str = DATA_PATH) -> dict:
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def _norm(s: str) -> str:
    """학과명/과목명 비교용 정규화: 공백·기호 제거, 로마숫자→아라비아, 소문자."""
    s = str(s)
    for a, b in {"Ⅰ": "1", "Ⅱ": "2", "Ⅲ": "3", "Ⅳ": "4", "ㆍ": "", "·": "", "•": ""}.items():
        s = s.replace(a, b)
    s = re.sub(r"[\s\-_,./:;'\"`~!?·&]", "", s)
    return s.lower()


def normalize_course(raw: str) -> str:
    """'캡스톤디자인(3학점) A+' 같은 입력에서 과목명만 남긴다."""
    s = str(raw).strip()
    s = re.sub(r"^\s*[\-\*•\d]+[.)]\s*", "", s)          # 목록 번호 "1." "-" 제거
    s = re.sub(r"\[[^\]]*\]", "", s)                       # [전공필수] 등
    s = re.sub(r"\([^)]*\)", "", s)                        # (3학점), (M&M) 등 괄호 제거
    s = re.sub(r"\s+(A\+|A0|A|B\+|B0|B|C\+|C0|C|D\+|D0|D|F|P|NP|S|U)\s*$", "", s, flags=re.I)
    s = re.sub(r"\s*\d+(\.\d+)?\s*학점\s*", "", s)
    return _norm(s)


def parse_course_list(text_or_list: str | list[str]) -> list[str]:
    """줄바꿈/쉼표로 구분된 과목 리스트를 정규화된 과목명 리스트로."""
    if isinstance(text_or_list, str):
        parts = re.split(r"[\n,，;]+", text_or_list)
    else:
        parts = list(text_or_list)
    out = []
    for p in parts:
        n = normalize_course(p)
        if n and n not in out:
            out.append(n)
    return out


def resolve_department(name: str, req: dict | None = None) -> str | None:
    req = req or load_requirements()
    q = _norm(name)
    if not q:
        return None
    for dept, info in req["departments"].items():
        cands = [dept] + info.get("aliases", [])
        if any(_norm(c) == q for c in cands):
            return dept
    for dept, info in req["departments"].items():
        cands = [dept] + info.get("aliases", [])
        if any(_norm(c) in q or q in _norm(c) for c in cands if len(_norm(c)) >= 2):
            return dept
    return None


def parse_year(value: Any) -> int | None:
    """'22학번', 22, '2022' → 2022"""
    m = re.search(r"\d{2,4}", str(value))
    if not m:
        return None
    y = int(m.group())
    if y < 100:
        y += 2000
    return y


def get_track(dept: str, year: int, req: dict | None = None) -> dict | None:
    req = req or load_requirements()
    for t in req["departments"][dept]["tracks"]:
        if t["years"][0] <= year <= t["years"][1]:
            return t
    return None


def get_common(year: int, req: dict | None = None) -> tuple[list[dict], str | None]:
    """대학공통 필수 항목. PDF 표는 2019학번부터만 있으므로 그 이전은 2019~2022 기준 + 경고."""
    req = req or load_requirements()
    for c in req["common"]:
        if c["years"][0] <= year <= c["years"][1]:
            return c["items"], None
    if year < 2019:
        return req["common"][0]["items"], (
            f"{year}학번의 대학공통 세부 항목은 안내문에 없어 2019~2022 기준으로 계산했습니다. "
            "多Dream비교과센터에 확인하세요."
        )
    return [], f"{year}학번 대학공통 기준이 안내문에 없습니다."


def get_requirement_items(dept: str, year: int, req: dict | None = None) -> dict:
    """학과·학번에 적용되는 전체 항목(대학공통 + 학과) 반환."""
    req = req or load_requirements()
    track = get_track(dept, year, req)
    common, warn = get_common(year, req)
    if track is None:
        avail = [f"{t['years'][0]}~{t['years'][1]}" for t in req["departments"][dept]["tracks"]]
        return {"error": f"{dept} {year}학번 기준은 안내문에 없습니다. (안내된 학번: {', '.join(avail)})"}
    # 구분 2단계: 대학공통(교과·비교과) / 학과기준(교과·졸업필수활동·비교과지정·비교과미지정)
    items = []
    overrides = track.get("common_overrides", {})
    for it in copy.deepcopy(common):
        it["group"] = "대학공통"
        it["kind"] = "교과" if it["type"] == "course" else "비교과"
        it.update(overrides.get(it["id"], {}))
        items.append(it)
    for it in copy.deepcopy(track["items"]):
        it["group"] = "학과기준"
        items.append(it)
    for it in items:
        it["section"] = section_label(it["group"], it["kind"])
    groups = {g: sum(i["points"] for i in items if i["group"] == g) for g in GROUPS}
    sections = {section_label(g, k): sum(i["points"] for i in items if i["group"] == g and i["kind"] == k)
                for g, k in SECTIONS}
    sections = {k: v for k, v in sections.items() if v}
    return {
        "department": dept,
        "year": year,
        "track_years": track["years"],
        "total": track["total"],
        "subtotals": track["subtotals"],
        "groups": groups,
        "sections": sections,
        "items": items,
        "unit_rules": req["departments"][dept].get("unit_rules", ""),
        "notes": req["departments"][dept].get("notes", []),
        "warnings": [warn] if warn else [],
    }


# ----------------------------------------------------------------------------
# 판정
# ----------------------------------------------------------------------------
@dataclass
class ItemResult:
    id: str
    name: str
    category: str
    group: str
    kind: str
    section: str
    required: float
    earned: float
    passed: bool
    detail: str = ""


@dataclass
class CheckResult:
    department: str
    year: int
    total_required: float
    total_earned: float
    passed: bool
    by_group: dict = field(default_factory=dict)     # {"대학공통": {...}, "학과기준": {...}}
    by_kind: dict = field(default_factory=dict)      # {"대학공통·교과": {...}, ...}
    by_category: dict = field(default_factory=dict)
    items: list = field(default_factory=list)
    missing: list = field(default_factory=list)
    unmatched_courses: list = field(default_factory=list)
    warnings: list = field(default_factory=list)
    notes: list = field(default_factory=list)

    def to_dict(self) -> dict:
        return asdict(self)


def _eval_item(item: dict, courses: list[str], act: Any, used: set) -> tuple[float, str]:
    t, pts = item["type"], item["points"]

    if t == "course":
        names = [_norm(m) for m in item.get("match", [item["name"]])]
        hit = next((c for c in courses if c in names), None)
        if hit:
            used.add(hit)
            return pts, "수강 리스트에서 확인"
        if act:
            return pts, "직접 체크"
        return 0, "미이수"

    if t == "check":
        return (pts, "체크됨") if act else (0, "미이수")

    if t == "all_of":
        parts = item["parts"]
        if act is True:
            done = list(parts)
        elif isinstance(act, dict):
            done = [p for p in parts if act.get(p)]
        elif isinstance(act, (list, tuple)):
            done = [p for p in parts if p in act]
        else:
            done = []
        ok = len(done) == len(parts)
        return (pts if ok else 0), f"{', '.join(done) or '없음'} 이수 ({', '.join(parts)} 모두 필요)"

    if t == "count":
        n = int(act or 0)
        earned = min(n * item["per"], pts)
        need = -(-pts // item["per"])  # ceil
        return earned, f"{n}{item.get('unit', '회')} 참여 (회당 {item['per']}점, 필요 {need}{item.get('unit', '회')})"

    if t == "choose":
        n_req = item["n"]
        each = pts / n_req
        if "pattern" in item:
            pat = _norm(item["pattern"])
            done = [c for c in courses if pat in c]
        else:
            opts = [_norm(o) for o in item["options"] + item.get("substitutes", [])]
            done = [c for c in courses if c in opts]
        extra = [normalize_course(x) for x in (act or [])] if isinstance(act, list) else []
        done = list(dict.fromkeys(done + extra))
        used.update(done)
        k = min(len(done), n_req)
        return k * each, f"{len(done)}과목 확인 ({', '.join(done) or '없음'}) / {n_req}과목 필요"

    if t == "points":
        if isinstance(act, (int, float)):
            total = float(act)
            return min(total, pts), f"입력 점수 {total:g}점"
        act = act or {}
        sub_vals = act.get("subitems", {}) if isinstance(act, dict) else {}
        total, parts = 0.0, []
        for s in item.get("subitems", []):
            v = sub_vals.get(s["name"])
            if not v:
                continue
            if s["mode"] == "check":
                p = s["points"]
            elif s["mode"] == "select":
                p = float(s["options"].get(v, v if isinstance(v, (int, float)) else 0))
            elif s["mode"] == "count":
                p = int(v) * s["per"]
            else:
                p = float(v)
            total += p
            parts.append(f"{s['name']} {p:g}점")
        other = float(act.get("other", 0) or 0) if isinstance(act, dict) else 0
        if other:
            total += other
            parts.append(f"{'기타' if item.get('subitems') else '취득'} {other:g}점")
        return min(total, pts), (", ".join(parts) or "인정 점수 없음") + f" (합 {total:g}점)"

    if t == "topcit_rule":
        a = act or {}
        att, lv = int(a.get("attempts", 0) or 0), int(a.get("level", 0) or 0)
        ok = att >= 5 or (att >= 3 and lv >= 2) or (att >= 1 and lv >= 3) or lv >= 4
        return (pts if ok else 0), f"응시 {att}회, 최고 Lv{lv} (기준: 응시5회 / 3회+Lv2 / 1회+Lv3 / Lv4)"

    raise ValueError(f"unknown type {t}")


def check_graduation(department: str, year: Any, courses: str | list[str] | None = None,
                     activities: dict | None = None, req: dict | None = None) -> CheckResult | dict:
    req = req or load_requirements()
    dept = resolve_department(department, req)
    if dept is None:
        return {"error": f"'{department}' 학과를 찾지 못했습니다. 가능: {', '.join(req['departments'])}"}
    y = parse_year(year)
    if y is None:
        return {"error": f"학번 '{year}'을(를) 해석하지 못했습니다."}
    spec = get_requirement_items(dept, y, req)
    if "error" in spec:
        return spec

    course_list = parse_course_list(courses or [])
    activities = activities or {}
    used: set = set()
    results: list[ItemResult] = []
    for it in spec["items"]:
        earned, detail = _eval_item(it, course_list, activities.get(it["id"]), used)
        results.append(ItemResult(it["id"], it["name"], it["category"], it["group"], it["kind"], it["section"], it["points"],
                                  round(earned, 2), earned >= it["points"] - 1e-9, detail))

    by_group = {g: {"required": sum(r.required for r in results if r.group == g),
                    "earned": round(sum(r.earned for r in results if r.group == g), 2)} for g in GROUPS}
    by_kind = {}
    for sec in spec["sections"]:
        by_kind[sec] = {"required": sum(r.required for r in results if r.section == sec),
                        "earned": round(sum(r.earned for r in results if r.section == sec), 2)}

    by_category = {}
    for c in CATEGORIES:
        by_category[c] = {"required": sum(r.required for r in results if r.category == c),
                          "earned": round(sum(r.earned for r in results if r.category == c), 2)}

    total_req = sum(r.required for r in results)
    total_ear = sum(r.earned for r in results)
    warnings = list(spec["warnings"])
    if abs(total_req - spec["total"]) > 1e-9:
        warnings.append(f"항목 합({total_req:g})이 안내문 합계({spec['total']})와 다릅니다.")

    missing = [
        {"name": r.name, "kind": r.kind, "group": r.group, "section": r.section, "category": r.category,
         "need": round(r.required - r.earned, 2), "detail": r.detail}
        for r in results if not r.passed
    ]
    return CheckResult(
        department=dept, year=y, total_required=total_req, total_earned=round(total_ear, 2),
        passed=all(r.passed for r in results), by_group=by_group, by_kind=by_kind, by_category=by_category,
        items=[asdict(r) for r in results], missing=missing,
        unmatched_courses=[c for c in course_list if c not in used],
        warnings=warnings, notes=spec["notes"] + ([spec["unit_rules"]] if spec["unit_rules"] else []),
    )


def format_result_text(res: CheckResult | dict) -> str:
    """사람이 읽는 요약 (CLI / LLM 컨텍스트용)."""
    if isinstance(res, dict) and "error" in res:
        return "⚠️ " + res["error"]
    lines = [
        f"[{res.department} {res.year}학번] 다면적 졸업인증제 판정",
        f"총점 {res.total_earned:g} / {res.total_required:g}  →  {'✅ 충족' if res.passed else '❌ 미충족'}",
        "구분별: " + " / ".join(
            f"{g} {gv['earned']:g}/{gv['required']:g} ("
            + ", ".join(f"{sec.split('·')[1]} {v['earned']:g}/{v['required']:g}"
                        for sec, v in res.by_kind.items() if sec.startswith(g)) + ")"
            for g, gv in res.by_group.items()),
        "인재상별: " + ", ".join(f"{k} {v['earned']:g}/{v['required']:g}" for k, v in res.by_category.items()),
    ]
    if res.missing:
        lines.append("부족한 항목:")
        for m in res.missing:
            lines.append(f"  - [{m['section']}/{m['category']}] {m['name']}: {m['need']:g}점 부족 ({m['detail']})")
    if res.unmatched_courses:
        lines.append("졸업인증 항목과 무관한 수강 과목: " + ", ".join(res.unmatched_courses))
    for w in res.warnings:
        lines.append("※ " + w)
    return "\n".join(lines)

In [ ]:
%%writefile rag/ingest.py
"""
문서 수집 → 청크 분할

두 종류의 청크를 만든다.
 1) 구조화 청크: data/requirements.json(PDF 표를 구조화한 것)을 학과·학번별 자연어 문단으로 변환.
    표 형태 PDF는 텍스트 추출 시 열이 뒤섞여 검색 품질이 나쁘므로, 표 1칸 = 청크 1개 수준으로 정리.
 2) 원문 청크: docs/ 폴더의 모든 문서(PDF, TXT, MD, DOCX)를 페이지/문단 단위로 분할.
    → '안내받은 다른 문서'는 docs/ 에 넣기만 하면 자동으로 포함된다.
"""
from __future__ import annotations

import hashlib
import re
from dataclasses import dataclass, asdict
from pathlib import Path

from .checker import load_requirements

ROOT = Path(__file__).resolve().parent.parent
DOCS_DIR = ROOT / "docs"
SUPPORTED = {".pdf", ".txt", ".md", ".docx"}


@dataclass
class Chunk:
    id: str
    text: str
    source: str          # 파일명 또는 'requirements.json'
    location: str        # 페이지 / 섹션
    department: str = "" # 메타데이터 필터·부스팅용
    years: tuple = ()

    def to_dict(self):
        return asdict(self)


def _cid(*parts) -> str:
    return hashlib.md5("|".join(map(str, parts)).encode()).hexdigest()[:12]


# ----------------------------------------------------------------------------
# 1) 구조화 청크
# ----------------------------------------------------------------------------
def _yr(y0, y1) -> str:
    return f"{y0}" if y0 == y1 else f"{y0}~{y1}"


def _item_line(it: dict) -> str:
    sec = it.get("section") or f"대학공통·{'교과' if it['type'] == 'course' else '비교과'}"
    s = f"- [{sec}/{it['category']}] {it['name']} ({it['points']:g}점)"
    if it["type"] == "count":
        s += f" — 회당 {it['per']}점, {-(-it['points'] // it['per'])}{it.get('unit', '회')} 필요"
    if it["type"] == "all_of":
        s += f" — {', '.join(it['parts'])} 모두 이수"
    if it["type"] == "choose":
        src = it.get("options") or [f"'{it['pattern']}' 포함 과목"]
        s += f" — {', '.join(src)} 중 {it['n']}과목"
        if it.get("substitutes"):
            s += f" (대체 인정: {', '.join(it['substitutes'])})"
    if it["type"] == "points" and it.get("subitems"):
        subs = []
        for x in it["subitems"]:
            if x["mode"] == "check":
                subs.append(f"{x['name']} {x['points']}점")
            elif x["mode"] == "select":
                subs.append(f"{x['name']}(" + ", ".join(f"{k} {v}점" for k, v in x["options"].items() if v) + ")")
            elif x["mode"] == "count":
                subs.append(f"{x['name']} 회당 {x['per']}점")
            else:
                subs.append(x["name"])
        s += " — 인정 예: " + ", ".join(subs)
    if it.get("note"):
        s += f" ※ {it['note']}"
    return s


def _category_line(dept: str, year: int, req: dict) -> str:
    from .checker import get_requirement_items, CATEGORIES
    items = get_requirement_items(dept, year, req)["items"]
    return "인재상별 배점(대학공통 포함): " + ", ".join(
        f"{c} {sum(i['points'] for i in items if i['category'] == c):g}점" for c in CATEGORIES)


def structured_chunks() -> list[Chunk]:
    req = load_requirements()
    meta = req["meta"]
    out: list[Chunk] = []

    out.append(Chunk(
        _cid("meta"),
        "다면적 졸업인증제 일반 안내\n"
        f"- 비교과 신청: {meta['apply_path']}\n"
        f"- 이수내역 확인: {meta['check_path']}\n"
        f"- 상담: 다면적졸업인증제에 대한 자세한 상담이 필요한 학생은 {meta['contact']}로 방문\n"
        "- 졸업인증 점수는 '대학공통 필수(45점)' + '학과 기준(교과, 비교과지정, 비교과미지정)'의 합계로 구성되며, "
        "인재상은 소통하는 공감인재 / 융합하는 창의인재 / 공헌하는 글로벌인재 3가지이다.\n"
        "- 적용 학과(SW융합대학): " + ", ".join(req["departments"]),
        meta["source"], "일반 안내"))

    for c in req["common"]:
        y0, y1 = c["years"]
        txt = (f"대학공통 필수 항목 ({_yr(y0, y1)}학번, 모든 학과 공통, 합계 45점, 교과·비교과로 구분)\n"
               + "\n".join(_item_line(i) for i in c["items"]))
        out.append(Chunk(_cid("common", y0), txt, meta["source"], f"대학공통 {_yr(y0, y1)}학번", "", (y0, y1)))

    from .checker import get_requirement_items
    for dept, info in req["departments"].items():
        for t in info["tracks"]:
            y0, y1 = t["years"]
            spec = get_requirement_items(dept, y0, req)
            sec = spec["sections"]
            parts = []
            for g in ("대학공통", "학과기준"):
                subs = [f"{k.split('·')[1]} {v:g}" for k, v in sec.items() if k.startswith(g)]
                parts.append(f"{g} {spec['groups'][g]:g}점({' + '.join(subs)})")
            dept_items = [i for i in spec["items"] if i["group"] == "학과기준"]
            moved = [i for i in dept_items if i["id"].startswith("c_")]
            txt = (f"{dept} {_yr(y0, y1)}학번 다면적 졸업인증제 기준 (SW융합대학)\n"
                   f"합계 {t['total']}점 = " + " + ".join(parts) + "\n"
                   "구분: 대학공통(교과·비교과) / 학과기준(교과·졸업필수활동·비교과지정·비교과미지정)\n"
                   + _category_line(dept, y0, req) + "\n"
                   "학과 기준 항목:\n" + "\n".join(_item_line(i) for i in dept_items))
            for m in moved:
                txt += f"\n※ {m['name']}은(는) 이 학과·학번에서 대학공통이 아닌 학과기준({m['kind']})으로 분류됩니다."
            if info.get("unit_rules"):
                txt += f"\n점수 산정: {info['unit_rules']}"
            for n in info.get("notes", []):
                txt += f"\n※ {n}"
            out.append(Chunk(_cid(dept, y0), txt, meta["source"], f"{dept} {_yr(y0, y1)}학번", dept, (y0, y1)))
    return out


# ----------------------------------------------------------------------------
# 2) 원문 문서 청크
# ----------------------------------------------------------------------------
def _split(text: str, size: int = 700, overlap: int = 120) -> list[str]:
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    if len(text) <= size:
        return [text] if text else []
    paras = re.split(r"\n\s*\n|\n(?=#)", text)
    chunks, cur = [], ""
    for p in paras:
        if len(cur) + len(p) + 1 <= size:
            cur = f"{cur}\n{p}" if cur else p
            continue
        if cur:
            chunks.append(cur)
        while len(p) > size:                 # 긴 문단은 고정 길이로 자르되 겹치게
            chunks.append(p[:size])
            p = p[size - overlap:]
        cur = (chunks[-1][-overlap:] + "\n" + p) if chunks else p
    if cur:
        chunks.append(cur)
    return chunks


def _guess_dept(text: str) -> str:
    req = load_requirements()
    found = [d for d in req["departments"] if d.replace("·", "") in text.replace("·", "").replace("ㆍ", "")]
    return found[0] if len(found) == 1 else ""


def read_document(path: Path) -> list[tuple[str, str]]:
    """(위치, 텍스트) 리스트."""
    ext = path.suffix.lower()
    if ext == ".pdf":
        from pypdf import PdfReader
        return [(f"p.{i + 1}", p.extract_text() or "") for i, p in enumerate(PdfReader(str(path)).pages)]
    if ext == ".md":   # 마크다운은 '## 제목' 단위로 나눠서 표·목록이 잘리지 않게
        text = path.read_text(encoding="utf-8", errors="ignore")
        parts = re.split(r"\n(?=## )", text)
        title = parts[0].strip().splitlines()[0].lstrip("# ").strip() if parts else path.stem
        out = []
        for sec in parts[1:] or parts:
            head = sec.strip().splitlines()[0].lstrip("# ").strip()
            out.append((head, f"[{title}]\n{sec.strip()}"))
        return out
    if ext == ".txt":
        return [("본문", path.read_text(encoding="utf-8", errors="ignore"))]
    if ext == ".docx":
        try:
            import docx  # python-docx
        except ImportError:
            print(f"[skip] {path.name}: python-docx 미설치")
            return []
        d = docx.Document(str(path))
        parts = [p.text for p in d.paragraphs]
        for tb in d.tables:
            for row in tb.rows:
                parts.append(" | ".join(c.text.strip() for c in row.cells))
        return [("본문", "\n".join(parts))]
    return []


def document_chunks(docs_dir: Path = DOCS_DIR) -> list[Chunk]:
    out = []
    for path in sorted(docs_dir.glob("**/*")):
        if path.suffix.lower() not in SUPPORTED or path.name.startswith((".", "_")):  # _README 등 안내 파일 제외
            continue
        for loc, text in read_document(path):
            for j, piece in enumerate(_split(text, size=1800 if path.suffix.lower() == ".md" else 700)):
                dept = _guess_dept(piece) if path.suffix.lower() == ".pdf" else ""  # 일반 안내 문서는 전 학과 공통
                out.append(Chunk(_cid(path.name, loc, j, piece[:50]), piece, path.name,
                                 f"{loc}" + (f" #{j + 1}" if j else ""), dept))
    return out


def build_chunks(docs_dir: Path = DOCS_DIR) -> list[Chunk]:
    return structured_chunks() + document_chunks(docs_dir)


if __name__ == "__main__":
    cs = build_chunks()
    print(f"총 {len(cs)}개 청크")
    for c in cs[:3]:
        print("-" * 60, f"\n[{c.source} / {c.location}]\n{c.text[:400]}")

In [ ]:
%%writefile rag/retriever.py
"""
하이브리드 검색기
  - 희소(Sparse): 문자 n-gram TF-IDF  → 한국어 형태소 분석기 없이도 '학술제발표' / '학술제 발표' 같은 변형을 잡는다.
  - 밀집(Dense) : OpenAI 호환 임베딩 (API 키가 있을 때만)
  - 메타데이터 부스팅: 질문에서 학과·학번을 인식하면 해당 청크 점수를 올리고 다른 학과 청크는 낮춘다.
"""
from __future__ import annotations

import hashlib
import os
import pickle
import re
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

from .checker import load_requirements, resolve_department, parse_year, _norm
from .ingest import Chunk, build_chunks, DOCS_DIR

INDEX_PATH = Path(__file__).resolve().parent.parent / "index" / "index.pkl"


def detect_profile(query: str) -> tuple[str | None, int | None]:
    """질문에서 학과/학번 추출. 예) 'AI빅데이터 22학번' → ('AI·빅데이터학과', 2022)"""
    req = load_requirements()
    dept = None
    q = _norm(query)
    best = 0
    for d, info in req["departments"].items():
        for a in [d] + info.get("aliases", []):
            na = _norm(a)
            if len(na) >= 2 and na in q and len(na) > best:
                dept, best = d, len(na)
    m = re.search(r"(20)?(\d{2})\s*학번", query)
    year = parse_year(m.group(0)) if m else None
    if year is None:
        m = re.search(r"\b(20[12]\d)\b", query)
        year = int(m.group(1)) if m else None
    return dept, year


class Retriever:
    def __init__(self, chunks: list[Chunk], embedder=None):
        self.chunks = chunks
        self.embedder = embedder
        self.tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True)
        self.sparse = self.tfidf.fit_transform([c.text for c in chunks])
        self.dense = None
        if embedder is not None:
            self.dense = self._normalize(np.array(embedder([c.text for c in chunks]), dtype=np.float32))

    @staticmethod
    def _normalize(m):
        return m / (np.linalg.norm(m, axis=-1, keepdims=True) + 1e-9)

    @property
    def fingerprint(self) -> str:
        return hashlib.md5("".join(c.id for c in self.chunks).encode()).hexdigest()

    def search(self, query: str, k: int = 5, department: str | None = None, year: int | None = None):
        d2, y2 = detect_profile(query)
        # 질문에 학과·학번이 적혀 있으면 그것을 우선, 없으면 사이드바 프로필 사용
        if d2 and d2 != department:
            department, year = d2, (y2 or None)
        else:
            department, year = department or d2, y2 or year

        sparse = (self.sparse @ self.tfidf.transform([query]).T).toarray().ravel()
        score = sparse
        if self.dense is not None and self.embedder is not None:
            qv = self._normalize(np.array(self.embedder([query])[0], dtype=np.float32))
            score = 0.6 * (self.dense @ qv) + 0.4 * sparse

        # 메타데이터 보정: 다른 학과/학번 자료는 감점, 같은 학과는 소폭 가점 (검색 점수를 뒤집지 않을 정도로)
        boost = np.zeros(len(self.chunks))
        for i, c in enumerate(self.chunks):
            if department and c.department:
                boost[i] += 0.04 if c.department == department else -0.08
            if year and c.years:
                boost[i] += 0.03 if c.years[0] <= year <= c.years[1] else -0.05
        final = score + boost

        # 학번이 인식되면 해당 학번의 대학공통 청크, 학과+학번이면 해당 학과 기준 청크를 반드시 포함
        pinned = []
        if year:
            for i, c in enumerate(self.chunks):
                if c.source and c.years and c.years[0] <= year <= c.years[1]:
                    if (c.department == "" ) or (department and c.department == department):
                        pinned.append(i)
        order = [i for i in np.argsort(-final) if i not in pinned]
        n_free = max(k - len(pinned), 2)
        free = order[:n_free]
        # 출처 다양성: 아직 결과에 없는 문서의 최고 점수 청크가 충분히 관련 있으면 포함
        if order:
            best = final[order[0]]
            shown = {self.chunks[i].source for i in pinned + free}
            for i in order[n_free:]:
                src = self.chunks[i].source
                if src not in shown and final[i] >= 0.6 * best:
                    free = free[:-1] + [i] if len(free) >= n_free else free + [i]
                    shown.add(src)
        top = sorted(pinned, key=lambda i: -final[i]) + sorted(free, key=lambda i: -final[i])
        return [(self.chunks[i], float(final[i])) for i in top if final[i] > 0 or i in pinned]


# ----------------------------------------------------------------------------
# 인덱스 빌드 / 캐시
# ----------------------------------------------------------------------------
def build_index(docs_dir: Path = DOCS_DIR, embedder=None, path: Path = INDEX_PATH, verbose=True) -> Retriever:
    chunks = build_chunks(docs_dir)
    r = Retriever(chunks, embedder)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump({"chunks": chunks, "dense": r.dense, "fp": r.fingerprint,
                     "docs": _docs_signature(docs_dir)}, f)
    if verbose:
        srcs = sorted({c.source for c in chunks})
        print(f"[index] {len(chunks)}개 청크, 임베딩={'O' if r.dense is not None else 'X (TF-IDF만)'}")
        print("[index] 문서:", ", ".join(srcs))
    return r


def _docs_signature(docs_dir: Path) -> str:
    items = sorted((p.name, p.stat().st_mtime, p.stat().st_size) for p in docs_dir.glob("**/*") if p.is_file())
    data_file = Path(__file__).resolve().parent.parent / "data" / "requirements.json"
    items.append(("requirements.json", data_file.stat().st_mtime, data_file.stat().st_size))
    return hashlib.md5(repr(items).encode()).hexdigest()


def load_or_build(docs_dir: Path = DOCS_DIR, embedder=None, path: Path = INDEX_PATH) -> Retriever:
    """docs/ 나 requirements.json 이 바뀌었으면 자동 재빌드."""
    if path.exists():
        with open(path, "rb") as f:
            saved = pickle.load(f)
        same_docs = saved.get("docs") == _docs_signature(docs_dir)
        same_mode = (saved["dense"] is not None) == (embedder is not None)
        if same_docs and same_mode:
            r = Retriever.__new__(Retriever)
            r.chunks, r.embedder, r.dense = saved["chunks"], embedder, saved["dense"]
            r.tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True)
            r.sparse = r.tfidf.fit_transform([c.text for c in r.chunks])
            return r
    return build_index(docs_dir, embedder, path, verbose=False)


def format_context(results) -> str:
    blocks = []
    for i, (c, s) in enumerate(results, 1):
        blocks.append(f"[문서 {i}] 출처: {c.source} / {c.location}\n{c.text}")
    return "\n\n".join(blocks)


if __name__ == "__main__":
    r = build_index()
    for q in ["AI빅데이터 22학번 졸업요건 알려줘", "TOPCIT 몇 번 봐야 해?", "비교과 이수내역 어디서 확인해?"]:
        print("\n### ", q)
        for c, s in r.search(q, k=3):
            print(f"  {s:.3f}  {c.source} / {c.location}")

In [ ]:
%%writefile rag/llm.py
"""
LLM 연결 + RAG 답변 생성 + Tool Calling

- OpenAI 호환 API 사용 (OPENAI_BASE_URL 로 Upstage Solar, Ollama, vLLM 등 교체 가능)
- 매 질문마다 검색(RAG) 결과를 컨텍스트로 넣고,
  LLM 이 필요하면 도구를 호출한다:
    search_documents   : 추가 검색
    get_requirements   : 학과·학번별 항목(id, 배점, 유형) 조회
    check_graduation   : 규칙 엔진으로 졸업요건 충족 여부 계산 (점수 계산은 LLM 이 하지 않음)
- API 키가 없으면 검색 결과만 보여주는 '오프라인 모드'로 동작
"""
from __future__ import annotations

import json
import os
from dataclasses import dataclass, field

from .checker import (check_graduation, format_result_text, get_requirement_items,
                      resolve_department, parse_year, load_requirements)
from .retriever import Retriever, format_context

CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4o-mini")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")

SYSTEM_PROMPT = """너는 순천향대학교 SW융합대학 학생을 돕는 '다면적 졸업인증제 상담 챗봇'이다.

규칙:
1. 반드시 [참고 문서]와 도구 결과에 근거해서만 답한다. 문서에 없는 내용은 추측하지 말고
   "안내문에 없는 내용"이라고 말한 뒤 多Dream비교과센터(향설 1관 RC306호) 문의를 권한다.
2. 답변에 사용한 근거는 [문서 n] 형태로 표시한다.
3. 학과·학번에 따라 기준이 다르므로, 학과/학번을 모르면 먼저 물어본다.
   (학생 프로필이 주어졌다면 그 학과·학번 기준으로 답한다.)
4. 졸업 충족 여부나 점수 계산은 절대 직접 계산하지 말고 check_graduation 도구를 사용한다.
   - 학생이 자신이 들은 과목/참여한 활동을 말하면: get_requirements 로 항목 id 를 확인한 뒤
     activities 를 채워 check_graduation 을 호출한다.
   - 과목명은 courses 리스트에 그대로 넣는다(과목 자동 인식).
   - 학생이 언급하지 않은 항목은 미이수로 두고, 결과에서 '부족한 항목'을 안내한다.
   - 자기주도상담·학과활동·전공상담 등 횟수형(count)은 참여 횟수를 넣는다 (회당 점수는 도구가 계산).
   - 세부 점수가 있는 항목(points)은 get_requirements 의 subitems 이름 그대로 {"subitems": {"SQL개발자(SQLD) 취득": true}} 처럼 넣는다.
5. 결과를 설명할 때: 총점/필요점수, 충족 여부, 부족한 항목과 채우는 방법(회당 점수 등)을 간결하게 정리한다.
6. 공식 확인은 종합정보시스템 → 학부학사행정 → 졸업 → 개인별졸업진도관리 → 비교과일람표 에서 하도록 안내한다.
7. 한국어로, 친근하지만 정확하게 답한다."""

TOOLS = [
    {"type": "function", "function": {
        "name": "search_documents",
        "description": "졸업인증제 안내 문서에서 관련 내용을 검색한다.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "검색어"}}, "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "get_requirements",
        "description": "학과·학번에 적용되는 졸업인증 항목 목록(id, 이름, 유형, 배점)을 조회한다. check_graduation 호출 전에 항목 id 확인용.",
        "parameters": {"type": "object", "properties": {
            "department": {"type": "string", "description": "학과명 (예: AI·빅데이터학과, 컴소)"},
            "admission_year": {"type": "integer", "description": "입학년도 4자리 (예: 2022)"}},
            "required": ["department", "admission_year"]}}},
    {"type": "function", "function": {
        "name": "check_graduation",
        "description": ("학생의 수강 과목과 비교과 활동으로 다면적 졸업인증 충족 여부를 계산한다. "
                        "activities 는 {항목id: 값}. 값 형식: course/check 항목=true, count 항목=참여횟수(정수), "
                        "points 항목=인정점수(숫자) 또는 {\"subitems\": {세부항목명: 값}}, choose 항목=[과목명], "
                        "all_of 항목=[이수한 부분명], topcit_rule 항목={attempts, level}."),
        "parameters": {"type": "object", "properties": {
            "department": {"type": "string"},
            "admission_year": {"type": "integer"},
            "courses": {"type": "array", "items": {"type": "string"}, "description": "수강(이수)한 과목명 리스트"},
            "activities": {"type": "object", "description": "{항목id: 값}"}},
            "required": ["department", "admission_year"]}}},
]


@dataclass
class Answer:
    text: str
    sources: list = field(default_factory=list)       # [(source, location, score)]
    tool_calls: list = field(default_factory=list)    # [(name, args, result_preview)]
    check_result: dict | None = None


# ----------------------------------------------------------------------------
# 클라이언트 / 임베딩
# ----------------------------------------------------------------------------
def get_client():
    key = os.getenv("OPENAI_API_KEY")
    if not key:
        return None
    try:
        from openai import OpenAI
    except ImportError:
        print("[warn] openai 패키지 미설치 → 오프라인 모드")
        return None
    base = os.getenv("OPENAI_BASE_URL") or None
    return OpenAI(api_key=key, base_url=base)


def make_embedder(client):
    """client 가 있으면 임베딩 함수 반환. USE_EMBEDDINGS=0 이면 끔."""
    if client is None or os.getenv("USE_EMBEDDINGS", "1") == "0":
        return None

    def embed(texts: list[str]) -> list[list[float]]:
        out = []
        for i in range(0, len(texts), 64):
            resp = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + 64])
            out.extend(d.embedding for d in resp.data)
        return out
    return embed


# ----------------------------------------------------------------------------
# 도구 실행
# ----------------------------------------------------------------------------
def _compact_requirements(dept: str, year: int) -> dict:
    d = resolve_department(dept)
    if not d:
        return {"error": f"학과 '{dept}'를 찾지 못함. 가능: {', '.join(load_requirements()['departments'])}"}
    spec = get_requirement_items(d, year)
    if "error" in spec:
        return spec
    items = []
    for it in spec["items"]:
        x = {"id": it["id"], "name": it["name"], "section": it["section"], "category": it["category"],
             "type": it["type"], "points": it["points"]}
        if it["type"] == "count":
            x["per"] = it["per"]
        if it["type"] == "choose":
            x["options"] = it.get("options") or f"'{it['pattern']}' 포함 과목"
            x["n"] = it["n"]
        if it["type"] == "all_of":
            x["parts"] = it["parts"]
        if it["type"] == "points" and it.get("subitems"):
            x["subitems"] = [{"name": s["name"], "mode": s["mode"],
                              **({"points": s["points"]} if "points" in s else {}),
                              **({"per": s["per"]} if "per" in s else {}),
                              **({"options": s["options"]} if "options" in s else {})} for s in it["subitems"]]
        items.append(x)
    return {"department": d, "year": year, "total": spec["total"], "groups": spec["groups"],
            "sections": spec["sections"], "items": items,
            "unit_rules": spec["unit_rules"], "notes": spec["notes"]}


def run_tool(name: str, args: dict, retriever: Retriever, state: dict) -> str:
    if name == "search_documents":
        res = retriever.search(args.get("query", ""), k=4,
                               department=state.get("department"), year=state.get("year"))
        state["sources"].extend(res)
        return format_context(res)
    if name == "get_requirements":
        return json.dumps(_compact_requirements(args["department"], int(args["admission_year"])), ensure_ascii=False)
    if name == "check_graduation":
        res = check_graduation(args["department"], args["admission_year"],
                               args.get("courses") or [], args.get("activities") or {})
        if isinstance(res, dict):
            return json.dumps(res, ensure_ascii=False)
        state["check_result"] = res.to_dict()
        return format_result_text(res)
    return json.dumps({"error": f"unknown tool {name}"})


# ----------------------------------------------------------------------------
# 답변 생성
# ----------------------------------------------------------------------------
def _profile_block(profile: dict | None, check_text: str | None) -> str:
    s = ""
    if profile and profile.get("department") and profile.get("year"):
        s += f"\n\n[학생 프로필] 학과: {profile['department']}, 학번: {profile['year']}"
    if check_text:
        s += f"\n\n[학생이 자가진단 탭에서 입력한 최근 판정 결과]\n{check_text}"
    return s


def answer(question: str, retriever: Retriever, client=None, history: list[dict] | None = None,
           profile: dict | None = None, check_text: str | None = None, k: int = 5,
           max_tool_rounds: int = 5) -> Answer:
    profile = profile or {}
    dept = resolve_department(profile.get("department", "")) if profile.get("department") else None
    year = parse_year(profile["year"]) if profile.get("year") else None

    retrieved = retriever.search(question, k=k, department=dept, year=year)
    context = format_context(retrieved)
    state = {"department": dept, "year": year, "sources": list(retrieved), "check_result": None}

    if client is None:
        return _offline_answer(question, retrieved, check_text)

    messages = [{"role": "system", "content": SYSTEM_PROMPT + _profile_block(profile, check_text)}]
    for h in (history or [])[-8:]:
        messages.append({"role": h["role"], "content": h["content"]})
    messages.append({"role": "user", "content": f"[참고 문서]\n{context}\n\n[질문]\n{question}"})

    trace = []
    for _ in range(max_tool_rounds):
        resp = client.chat.completions.create(model=CHAT_MODEL, messages=messages, tools=TOOLS,
                                              temperature=0.1)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return Answer(msg.content or "", _dedup_sources(state["sources"]), trace, state["check_result"])
        messages.append({"role": "assistant", "content": msg.content or "",
                         "tool_calls": [tc.model_dump() for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            try:
                args = json.loads(tc.function.arguments or "{}")
                out = run_tool(tc.function.name, args, retriever, state)
            except Exception as e:  # 도구 오류도 LLM 에 돌려줘서 스스로 수정하게
                args, out = {}, f"도구 실행 오류: {e}"
            trace.append((tc.function.name, args, out[:600]))
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})

    resp = client.chat.completions.create(model=CHAT_MODEL, messages=messages, temperature=0.1)
    return Answer(resp.choices[0].message.content or "", _dedup_sources(state["sources"]), trace,
                  state["check_result"])


def _dedup_sources(results):
    seen, out = set(), []
    for c, s in results:
        key = (c.source, c.location)
        if key not in seen:
            seen.add(key)
            out.append((c.source, c.location, round(s, 3)))
    return out


def _offline_answer(question, retrieved, check_text) -> Answer:
    parts = ["🔌 **오프라인 모드** (OPENAI_API_KEY 없음) — LLM 생성 없이 검색된 근거만 보여줍니다.\n"]
    if check_text and any(w in question for w in ["부족", "충족", "졸업", "가능", "남은"]):
        parts.append("**최근 자가진단 결과**\n```\n" + check_text + "\n```")
    for i, (c, s) in enumerate(retrieved[:3], 1):
        parts.append(f"**[문서 {i}] {c.source} / {c.location}** (유사도 {s:.2f})\n```\n{c.text[:900]}\n```")
    return Answer("\n\n".join(parts), _dedup_sources(retrieved))

## 3. 인덱스 구축 & 검증

In [ ]:
import importlib, rag.checker, rag.ingest, rag.retriever, rag.llm
for m in (rag.checker, rag.ingest, rag.retriever, rag.llm): importlib.reload(m)
from rag.checker import *
from rag.retriever import build_index
from rag.llm import answer, get_client, make_embedder

# (검증) 모든 학과·학번 항목 합 == PDF 표의 소계/합계
req = load_requirements(); bad = 0
for d, info in req["departments"].items():
    for t in info["tracks"]:
        items = get_requirement_items(d, t["years"][0], req)["items"]
        ok = sum(i["points"] for i in items) == t["total"] and all(
            sum(i["points"] for i in items if i["kind"] == k) == t["subtotals"][k] for k in KINDS)
        bad += not ok
        print(("✅" if ok else "❌"), d, f"{t['years'][0]}~{t['years'][1]}", t["total"], "점")
print("데이터 검증:", "모두 일치" if bad == 0 else f"{bad}건 불일치")

client = get_client()
retriever = build_index(embedder=make_embedder(client))

In [ ]:
# 검색 테스트
for q in ["AI빅데이터 22학번 졸업요건", "권장 자격증 뭐 있어?", "AI·SW전공역량졸업인증제 200점 어떻게 채워?", "다면적졸업인증만 채우면 졸업 가능해?"]:
    print("\nQ:", q)
    for c, s in retriever.search(q, k=3, department="AI·빅데이터학과", year=2022):
        print(f"   {s:.3f}  {c.source} / {c.location}")

## 4. 졸업 판정 예시 — AI·빅데이터학과 22학번

In [ ]:
res = check_graduation(
    "AI빅데이터", "22학번",
    courses="""인간사랑 A+
현대기본영어(2학점) B+
봉사학습
예비창업설계""",
    activities={"c_asp": ["1학기", "2학기"],     # ASP 1·2학기 모두 이수
                "c_selfcounsel": 3,                # 자기주도상담 3회 → 3점 (회당 1점)
                "c_gradact": 10,                   # 졸업필수활동 취득 점수 10/15
                "dept_act": 3,                     # 학과활동 3회 → 3점
                "cert_contest": {"subitems": {"SQL개발자(SQLD) 취득": True}},   # 권장 자격증 6점
                "aisw": True, "free_global": 6})
print(format_result_text(res))

## 5. RAG 질의응답 예시

In [ ]:
profile = {"department": "AI·빅데이터학과", "year": 2022}
for q in ["학과활동은 몇 번 참여해야 해?", "AI·SW전공역량졸업인증제 200점은 어떻게 채워?",
          "인간사랑, 현대기본영어, 예비창업설계 들었고 학과활동 3번, SQLD 땄어. 졸업 가능해? 뭐가 부족해?"]:
    a = answer(q, retriever, client, profile=profile)
    print("Q:", q, "\n")
    print(a.text[:1500])
    if a.tool_calls: print("\n🛠️ 도구:", [t[0] for t in a.tool_calls])
    print("📎 근거:", [f"{s}/{l}" for s, l, _ in a.sources[:3]], "\n" + "=" * 80)

## 6. 챗봇 실행 (Gradio)
아래 셀 실행 → 화면이 나타나고 `https://xxxx.gradio.live` **공개 링크**(72시간 유효)가 출력됩니다.
- 💬 **졸업요건 Q&A**: 학과/학번 기준으로 문서 근거와 함께 답변, 과목·활동을 말하면 졸업 판정
- ✅ **졸업 자가진단**: 활동 체크 + 수강 과목 리스트 → 충족 여부/부족 항목

In [ ]:
%%writefile gradio_app.py
"""
Gradio 버전 UI (Colab / Hugging Face Spaces 용)
  - Colab:  노트북 마지막 셀에서 build_demo().launch(share=True)
  - 로컬:   python gradio_app.py
"""
from __future__ import annotations

import gradio as gr

from rag.checker import load_requirements, get_requirement_items, check_graduation, format_result_text
from rag.retriever import load_or_build
from rag.llm import answer, get_client, make_embedder

REQ = load_requirements()
DEPTS = list(REQ["departments"])


def _text(content) -> str:
    """Gradio 버전에 따라 메시지 content 가 문자열 또는 블록 리스트일 수 있음."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in content)
    if isinstance(content, dict):
        return content.get("text", "")
    return str(content)


def _result_markdown(res) -> str:
    if isinstance(res, dict):
        return "⚠️ " + res["error"]
    head = "## ✅ 졸업인증 충족" if res.passed else "## ❌ 아직 미충족"
    lines = [head, f"**총점 {res.total_earned:g} / {res.total_required:g}점**", "",
             "| 구분 | 취득 | 필요 |", "|---|---|---|"]
    for g, gv in res.by_group.items():
        lines.append(f"| **{g}** | **{gv['earned']:g}** | **{gv['required']:g}** |")
        lines += [f"| └ {k.split('·')[1]} | {v['earned']:g} | {v['required']:g} |" for k, v in res.by_kind.items() if k.startswith(g)]
    lines += ["", "| 인재상 | 취득 | 필요 |", "|---|---|---|"]
    lines += [f"| {k} | {v['earned']:g} | {v['required']:g} |" for k, v in res.by_category.items()]
    if res.missing:
        lines += ["", "### 부족한 항목", "| 구분 | 인재상 | 항목 | 부족 | 현재 상태 |", "|---|---|---|---|---|"]
        lines += [f"| {m['section']} | {m['category']} | {m['name']} | {m['need']:g}점 | {m['detail']} |" for m in res.missing]
    if res.unmatched_courses:
        lines += ["", "ℹ️ 졸업인증 항목과 매칭되지 않은 과목: " + ", ".join(res.unmatched_courses)]
    lines += [""] + [f"※ {w}" for w in res.warnings + res.notes]
    lines += ["", "_공식 확인: 종합정보시스템 → 학부학사행정 → 졸업 → 개인별졸업진도관리 → 비교과일람표_"]
    return "\n".join(lines)


def build_demo(default_dept: str = "AI·빅데이터학과", default_year: int = 2022):
    client = get_client()
    retriever = load_or_build(embedder=make_embedder(client))
    mode = "LLM 연결됨" if client else "오프라인 모드 (API 키 없음 → 검색 결과만 표시)"
    sources = sorted({c.source for c in retriever.chunks})

    with gr.Blocks(title="SW융합대학 졸업요건 챗봇") as demo:
        gr.Markdown(f"# 🎓 SW융합대학 다면적 졸업인증제 챗봇\n"
                    f"상태: **{mode}** · 참고 문서: {', '.join(sources)} · 청크 {len(retriever.chunks)}개")
        with gr.Row():
            dept = gr.Dropdown(DEPTS, value=default_dept, label="학과")
            year = gr.Number(value=default_year, precision=0, label="학번(입학년도)")
        check_state = gr.State("")

        # ------------------------------------------------------------- Q&A 탭
        with gr.Tab("💬 졸업요건 Q&A"):
            try:
                chatbot = gr.Chatbot(type="messages", height=480, label="챗봇")
            except TypeError:  # Gradio 6+ 는 messages 형식이 기본
                chatbot = gr.Chatbot(height=480, label="챗봇")
            msg = gr.Textbox(placeholder="예) 학과활동은 몇 번 해야 해? / 인간사랑, 예비창업설계 들었는데 졸업 가능해?",
                             label="질문", lines=2)
            with gr.Row():
                send = gr.Button("보내기", variant="primary")
                clear = gr.Button("대화 초기화")
            gr.Examples(["내 학번 졸업인증 기준 전체 알려줘", "학과활동은 몇 번 참여해야 해?",
                         "권장 자격증은 뭐가 있어?", "AI·SW전공역량졸업인증제 200점은 어떻게 채워?",
                         "인간사랑, 현대기본영어, 봉사학습, 예비창업설계 들었고 학과활동 3번, SQLD 땄어. 졸업 가능해?"],
                        inputs=msg)

            def respond(m, hist, d, y, chk):
                if not m.strip():
                    return "", hist
                hist = hist or []
                h = [{"role": x["role"], "content": _text(x["content"])} for x in hist if isinstance(x, dict)]
                try:
                    a = answer(m, retriever, client, h, {"department": d, "year": int(y)}, chk or None)
                    text = a.text
                    if a.tool_calls:
                        text += "\n\n<sub>🛠️ 도구 호출: " + " → ".join(t[0] for t in a.tool_calls) + "</sub>"
                    if a.sources:
                        text += "\n\n<sub>📎 근거: " + " · ".join(f"{s} / {l}" for s, l, _ in a.sources[:4]) + "</sub>"
                except Exception as e:
                    text = f"⚠️ 오류: {e}"
                return "", hist + [{"role": "user", "content": m}, {"role": "assistant", "content": text}]

            msg.submit(respond, [msg, chatbot, dept, year, check_state], [msg, chatbot])
            send.click(respond, [msg, chatbot, dept, year, check_state], [msg, chatbot])
            clear.click(lambda: [], None, chatbot)

        # ------------------------------------------------------------- 자가진단 탭
        with gr.Tab("✅ 졸업 자가진단"):
            with gr.Row():
                with gr.Column(scale=2):
                    courses = gr.Textbox(lines=12, label="📘 수강한 교과목 리스트 (한 줄에 하나, 성적표 붙여넣기 가능)",
                                         placeholder="인간사랑\n현대기본영어 A+\n봉사학습(1학점)\n예비창업설계")
                    result = gr.Markdown("왼쪽/오른쪽을 입력하고 **판정하기**를 누르세요.")
                with gr.Column(scale=3):
                    @gr.render(inputs=[dept, year])
                    def build_items(d, y):
                        try:
                            spec = get_requirement_items(d, int(y))
                        except Exception as e:
                            gr.Markdown(f"⚠️ {e}")
                            return
                        if "error" in spec:
                            gr.Markdown("⚠️ " + spec["error"])
                            return
                        gr.Markdown(f"### 🏃 {d} {int(y)}학번 · 기준 {spec['total']}점")
                        comps, meta = [], []
                        for sec, sec_pts in spec["sections"].items():
                            items = [i for i in spec["items"] if i["section"] == sec]
                            gr.Markdown(f"**{sec}** ({sec_pts:g}점)")
                            for it in items:
                                lab = f"{it['name']} ({it['points']:g}점)"
                                t = it["type"]
                                if t in ("course", "check"):
                                    info = "수강 리스트에 있으면 자동 인정" if t == "course" else it.get("note")
                                    comps.append(gr.Checkbox(label=lab, info=info))
                                    meta.append((it["id"], "bool"))
                                elif t == "all_of":
                                    comps.append(gr.CheckboxGroup(it["parts"], label=f"{lab} — 이수한 부분 (모두 필요)",
                                                                  info=it.get("note")))
                                    meta.append((it["id"], "list"))
                                elif t == "count":
                                    comps.append(gr.Number(0, precision=0, minimum=0,
                                                           label=f"{lab} — 참여 횟수 (회당 {it['per']}점)"))
                                    meta.append((it["id"], "int"))
                                elif t == "choose":
                                    opts = (it.get("options") or []) + it.get("substitutes", [])
                                    if opts:
                                        comps.append(gr.CheckboxGroup(opts, label=f"{lab} — {it['n']}과목 필요"))
                                        meta.append((it["id"], "list"))
                                    else:
                                        gr.Markdown(f"- {lab}: 수강 리스트에서 '{it['pattern']}' 포함 과목을 셉니다.")
                                elif t == "points" and it.get("subitems"):
                                    gr.Markdown(f"**{lab}**" + (f" — {it['note']}" if it.get("note") else ""))
                                    for s_ in it["subitems"]:
                                        if s_["mode"] == "check":
                                            comps.append(gr.Checkbox(label=f"{s_['name']} (+{s_['points']}점)"))
                                        elif s_["mode"] == "count":
                                            comps.append(gr.Number(0, precision=0, minimum=0, label=f"{s_['name']} 횟수 (회당 {s_['per']}점)"))
                                        elif s_["mode"] == "select":
                                            comps.append(gr.Dropdown(list(s_["options"]), value=list(s_["options"])[0], label=s_["name"]))
                                        else:
                                            comps.append(gr.Number(0, minimum=0, label=s_["name"]))
                                        meta.append((it["id"], ("sub", s_["name"])))
                                    comps.append(gr.Number(0, minimum=0, label="기타 비교과 인정 점수"))
                                    meta.append((it["id"], "other"))
                                elif t == "points":
                                    ex = ", ".join(
                                        f"{s['name']}" + (f" {s['points']}점" if s.get("points") else
                                                          f" 회당 {s['per']}점" if s.get("per") else "")
                                        for s in it.get("subitems", []))
                                    comps.append(gr.Number(0, minimum=0, label=f"{lab} — 취득 점수",
                                                           info=it.get("note") or ("예: " + ex if ex else "비교과일람표의 인정 점수")))
                                    meta.append((it["id"], "float"))
                                elif t == "topcit_rule":
                                    comps.append(gr.Number(0, precision=0, minimum=0, label="TOPCIT 응시 횟수"))
                                    meta.append((it["id"], "topcit_att"))
                                    comps.append(gr.Dropdown([0, 1, 2, 3, 4, 5], value=0, label="TOPCIT 최고 수준"))
                                    meta.append((it["id"], "topcit_lv"))
                        btn = gr.Button("판정하기", variant="primary")

                        def run(course_text, *vals):
                            acts = {}
                            for (iid, kind), v in zip(meta, vals):
                                if kind == "bool":
                                    acts[iid] = bool(v)
                                elif kind == "int":
                                    acts[iid] = int(v or 0)
                                elif kind == "float":
                                    acts[iid] = float(v or 0)
                                elif kind == "list":
                                    acts[iid] = list(v or [])
                                elif isinstance(kind, tuple):
                                    acts.setdefault(iid, {"subitems": {}, "other": 0})["subitems"][kind[1]] = v
                                elif kind == "other":
                                    acts.setdefault(iid, {"subitems": {}, "other": 0})["other"] = float(v or 0)
                                elif kind == "topcit_att":
                                    acts.setdefault(iid, {})["attempts"] = int(v or 0)
                                elif kind == "topcit_lv":
                                    acts.setdefault(iid, {})["level"] = int(v or 0)
                            res = check_graduation(d, int(y), course_text or "", acts)
                            return _result_markdown(res), format_result_text(res)

                        btn.click(run, [courses] + comps, [result, check_state])
            gr.Markdown("판정 후 **Q&A 탭**에서 \"부족한 부분 어떻게 채워?\"라고 물으면 판정 결과를 참고해 답합니다.")
    return demo


if __name__ == "__main__":
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    build_demo().launch()

In [ ]:
import importlib, gradio_app; importlib.reload(gradio_app)
demo = gradio_app.build_demo(default_dept="AI·빅데이터학과", default_year=2022)
demo.launch(share=True, debug=False)